# Step 1: Data Preprocessing and Exploratory Behavioral Analysis

In [1]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 1: Data Preprocessing and Exploratory Behavioral Analysis
=============================================================================
Pipeline Position: ENTRY POINT (no upstream dependencies)
Downstream Consumers: Step 2a (reads 'hddm_data_unfair.csv' and
                      'data_fingerprint.json')
=============================================================================
"""

import os
import json
import hashlib
from datetime import datetime
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter, PercentFormatter

# =============================================================================
# GLOBAL CONSTANTS & AESTHETICS
# =============================================================================
sns.set_theme(style="ticks", palette="colorblind")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'legend.frameon': False,
    'pdf.fonttype': 42
})

ACC_COLOR = "#004D40"
REJ_COLOR = "#4A148C"
ACC_ALPHA = 0.52
REJ_ALPHA = 0.75

# Reaction Time (RT) boundaries in milliseconds.
# Minimum RT set to 300ms to accommodate complex social cognition
# processing time in the Ultimatum Game paradigm.
RT_MIN_MS = 300
RT_MAX_MS = 3000

# Behavioral response coding in the original experimental data
RESPONSE_ACCEPT = 1
RESPONSE_REJECT = 2
RESPONSE_NONE = 0

# HDDM response coding boundary mapping:
# Upper boundary (1) = Accept; Lower boundary (0) = Reject
HDDM_ACCEPT = 1
HDDM_REJECT = 0


# =============================================================================
# LOGGING SETUP
# =============================================================================
def setup_logger(log_file: str = "step1_data_preparation.log") -> logging.Logger:
    """Initialize logging configuration for process tracking."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s %(levelname)s: %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)


# =============================================================================
# DATA FINGERPRINTING & LINEAGE
# =============================================================================
def compute_file_hash(filepath: str) -> str:
    """
    Computes the SHA-256 cryptographic hash of a specified file.
    Returns an empty string if the file is inaccessible.
    """
    if not os.path.exists(filepath):
        return ""
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()


def generate_data_fingerprint(
    df: pd.DataFrame,
    source_filepath: str,
    output_filepath: str,
    logger: logging.Logger
) -> None:
    """
    Extracts structural metadata and computes cryptographic hashes for
    both the source data and the final analytical matrix. Serializes the
    artifact to a JSON file for downstream pipeline validation.
    """
    logger.info("--- Generating Cryptographic Data Fingerprint ---")

    source_hash = compute_file_hash(source_filepath)
    output_hash = compute_file_hash(output_filepath)

    fingerprint = {
        "data_file_name": os.path.basename(output_filepath),
        "data_file_hash": output_hash,
        "source_file_hash": source_hash,
        "n_rows": int(len(df)),
        "n_subjects": int(df['subj_idx'].nunique()),
        "emotion_levels": sorted(df['emotion'].unique().tolist()),
        "response_levels": sorted(df['response'].unique().tolist()),
        "rt_min_sec": float(df['rt'].min()),
        "rt_max_sec": float(df['rt'].max()),
        "created_at_utc": datetime.utcnow().isoformat() + "Z",
        "preprocessing_signature": {
            "task_scope": "unfair_only",
            "emotion_recode": "enj->rew",
            "exclusion_rule_rt_min_ms": RT_MIN_MS,
            "exclusion_rule_rt_max_ms": RT_MAX_MS,
            "response_mapping": "Accept=1, Reject=0"
        }
    }

    fingerprint_path = os.path.join(
        os.path.dirname(output_filepath) or ".", "data_fingerprint.json"
    )
    with open(fingerprint_path, 'w', encoding='utf-8') as f:
        json.dump(fingerprint, f, indent=4)

    logger.info(f"Data fingerprint serialized to '{fingerprint_path}'.")
    logger.info(f"Target Lineage Hash: {output_hash[:16]}...")


# =============================================================================
# MODULE 1: RESPONSE CODING AUDIT
# =============================================================================
def audit_response_coding(
    df: pd.DataFrame, logger: logging.Logger
) -> pd.DataFrame:
    """
    Strictly audits the mapping between original button presses and
    HDDM boundary coding. Enforces bijective mapping:
      Original Accept (1) -> HDDM Upper Boundary (1)
      Original Reject (2) -> HDDM Lower Boundary (0)
    """
    logger.info("--- Executing Response Coding Audit ---")

    orig_accept = (df['reaction'] == RESPONSE_ACCEPT).sum()
    orig_reject = (df['reaction'] == RESPONSE_REJECT).sum()

    response_mapping = {RESPONSE_ACCEPT: HDDM_ACCEPT,
                        RESPONSE_REJECT: HDDM_REJECT}
    df['response_hddm'] = df['reaction'].map(response_mapping)

    hddm_accept = (df['response_hddm'] == HDDM_ACCEPT).sum()
    hddm_reject = (df['response_hddm'] == HDDM_REJECT).sum()

    if orig_accept != hddm_accept or orig_reject != hddm_reject:
        raise ValueError("CRITICAL: Response mapping mismatch detected!")

    if df['response_hddm'].isnull().any():
        unmapped = df.loc[df['response_hddm'].isnull(), 'reaction'].unique()
        raise ValueError(f"Unmapped response values detected: {unmapped}")

    # [FIX-1.2] Explicit binary validation
    unique_resp = set(df['response_hddm'].unique())
    if not unique_resp.issubset({0, 1}):
        raise ValueError(
            f"HDDM response column contains non-binary values: {unique_resp}"
        )

    audit_data = [{
        'Original_Response': 'Accept (1)', 'Original_Count': orig_accept,
        'HDDM_Boundary': 'Upper (1)', 'HDDM_Count': hddm_accept
    }, {
        'Original_Response': 'Reject (2)', 'Original_Count': orig_reject,
        'HDDM_Boundary': 'Lower (0)', 'HDDM_Count': hddm_reject
    }]

    pd.DataFrame(audit_data).to_csv("response_coding_audit.csv", index=False)
    logger.info(
        "Response coding audit passed and exported to "
        "'response_coding_audit.csv'."
    )
    return df


# =============================================================================
# MODULE 2: FAIR-CEILING DIAGNOSTICS & ROBUSTNESS CHECKS
# =============================================================================
def generate_fair_ceiling_diagnostics(
    df_valid: pd.DataFrame, logger: logging.Logger
):
    """
    Calculates rejection rates across Fair and Unfair conditions to
    justify targeting Unfair trials exclusively. Ceiling effects in
    Fair offers (near 100% acceptance) would violate DDM variance
    assumptions and cause MCMC convergence failure.
    """
    logger.info("--- Generating Fair-Ceiling Diagnostics ---")

    df_valid = df_valid.copy()
    df_valid['Condition_Type'] = np.where(
        df_valid['Offers_You'] <= 2, 'Unfair (9:1, 8:2)',
        np.where(
            df_valid['Offers_You'] >= 4, 'Fair (5:5, 6:4)', 'Intermediate'
        )
    )

    df_target = df_valid[
        df_valid['Condition_Type'].isin(
            ['Unfair (9:1, 8:2)', 'Fair (5:5, 6:4)']
        )
    ]

    summary = df_target.groupby(['Condition_Type', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()

    summary.to_csv("fair_ceiling_diagnostics.csv", index=False)
    logger.info(
        "Fair-ceiling behavior summary exported to "
        "'fair_ceiling_diagnostics.csv'."
    )

    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=summary, x='emotion', y='Rejection_Rate',
        hue='Condition_Type',
        palette=['#4A148C', '#900C3F'], alpha=0.85
    )
    plt.title("Empirical Rejection Rates: Fair vs. Unfair Offers", pad=15)
    plt.ylabel("Probability of Rejection")
    plt.xlabel("Emotion Condition")
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.ylim(0, 1.05)
    plt.legend(title="Offer Type", loc='upper left',
               bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig("fair_unfair_rejection_rate.pdf", dpi=300)
    plt.close()


def generate_offer_ratio_robustness(
    df_unfair: pd.DataFrame, logger: logging.Logger
):
    """
    Validates whether 9:1 and 8:2 ratios behave similarly enough
    to merge into a single 'unfair' category for HDDM estimation.
    """
    logger.info("--- Generating 9:1 vs 8:2 Robustness Check ---")

    summary = df_unfair.groupby(['Offers_You', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()

    summary['Offer_Ratio'] = summary['Offers_You'].map({1: '9:1', 2: '8:2'})
    summary.to_csv("unfair_offer_ratio_behavior_summary.csv", index=False)
    logger.info(
        "Offer ratio robustness check exported to "
        "'unfair_offer_ratio_behavior_summary.csv'."
    )


# =============================================================================
# [FIX-1.1] MODULE 3: PER-SUBJECT TRIAL COUNT AUDIT
# =============================================================================
def audit_trial_counts(
    hddm_df: pd.DataFrame, logger: logging.Logger
) -> pd.DataFrame:
    """
    Generates a subject x condition trial count matrix to verify that
    each experimental cell meets the 30-40 trial minimum required for
    stable four-parameter DDM estimation without across-trial variability
    (Lerche & Voss, 2016, Behavior Research Methods).

    Flags subjects with fewer than 20 trials in any condition as
    candidates for exclusion in sensitivity analyses.
    """
    logger.info("--- Auditing Per-Subject Trial Counts ---")

    trial_counts = hddm_df.groupby(
        ['subj_idx', 'emotion']
    ).size().reset_index(name='n_trials')

    pivot = trial_counts.pivot(
        index='subj_idx', columns='emotion', values='n_trials'
    ).fillna(0).astype(int)

    pivot['min_across_conditions'] = pivot.min(axis=1)
    pivot['total_trials'] = pivot.drop(
        columns='min_across_conditions'
    ).sum(axis=1)

    # Flag subjects below minimum threshold
    MINIMUM_TRIALS_PER_CONDITION = 20
    pivot['below_threshold'] = (
        pivot['min_across_conditions'] < MINIMUM_TRIALS_PER_CONDITION
    )

    n_flagged = pivot['below_threshold'].sum()
    if n_flagged > 0:
        logger.warning(
            f"  {n_flagged} subject(s) have fewer than "
            f"{MINIMUM_TRIALS_PER_CONDITION} trials in at least one "
            f"condition. Consider exclusion in sensitivity analyses."
        )
    else:
        logger.info(
            f"  All subjects meet the minimum trial threshold "
            f"({MINIMUM_TRIALS_PER_CONDITION} trials/condition)."
        )

    pivot.to_csv("subject_trial_count_audit.csv")
    logger.info("Per-subject trial counts exported to "
                "'subject_trial_count_audit.csv'.")

    # Summary statistics
    logger.info(f"  Trial count range: "
                f"{int(pivot['min_across_conditions'].min())} - "
                f"{int(pivot['min_across_conditions'].max())} "
                f"(min across conditions per subject)")
    logger.info(f"  Grand mean trials per cell: "
                f"{trial_counts['n_trials'].mean():.1f}")

    return trial_counts


# =============================================================================
# DATA LOADING, FILTERING AND HDDM INGESTION
# =============================================================================
def load_and_filter_data(
    filepath: str, logger: logging.Logger
) -> pd.DataFrame:
    """
    Loads raw data, applies exclusion criteria, and standardizes labels.
    Exclusion pipeline:
      1. Standardize emotion labels ('enj' -> 'rew')
      2. Remove omitted responses (reaction == 0)
      3. Remove fast RTs (< 300ms)
      4. Remove slow RTs (> 3000ms)
      5. Select unfair offers only (Offers_You in {1, 2})
    """
    logger.info(f"Loading raw data from {filepath}")

    df = pd.read_csv(filepath)
    logger.info(f"Initial raw data dimensions: {df.shape}")

    exclusion_log = [{'Stage': 'Total_Initial_Trials', 'Count': len(df)}]

    # 1. Standardize emotion legacy labels: 'enj' -> 'rew'
    df['emotion'] = (
        df['emotion'].astype(str).str.strip().replace({'enj': 'rew'})
    )

    # 2. Missing responses
    omitted_mask = df['reaction'] == RESPONSE_NONE
    df_valid_resp = df[~omitted_mask]
    exclusion_log.append({
        'Stage': 'Omitted_Responses', 'Count': omitted_mask.sum()
    })

    # 3. RT boundaries (lower)
    fast_mask = df_valid_resp['RT'] < RT_MIN_MS
    df_valid_rt_low = df_valid_resp[~fast_mask]
    exclusion_log.append({
        'Stage': 'RT_Too_Fast', 'Count': fast_mask.sum()
    })

    # 4. RT boundaries (upper)
    slow_mask = df_valid_rt_low['RT'] > RT_MAX_MS
    df_valid_rt = df_valid_rt_low[~slow_mask]
    exclusion_log.append({
        'Stage': 'RT_Too_Slow', 'Count': slow_mask.sum()
    })

    # Trigger diagnostic before filtering fairness
    generate_fair_ceiling_diagnostics(df_valid_rt, logger)

    # 5. Target Unfair Condition Selection (Offers_You == 1 or 2)
    fair_mask = ~df_valid_rt['Offers_You'].isin([1, 2])
    df_unfair = df_valid_rt[~fair_mask].copy()
    exclusion_log.append({
        'Stage': 'Non_Unfair_Offers_Excluded', 'Count': fair_mask.sum()
    })
    exclusion_log.append({
        'Stage': 'Final_Retained_Unfair_Trials', 'Count': len(df_unfair)
    })

    pd.DataFrame(exclusion_log).to_csv(
        "exclusion_summary_flow.csv", index=False
    )
    logger.info(f"Retained trials (Unfair conditions only): {len(df_unfair)}")

    df_unfair = audit_response_coding(df_unfair, logger)
    generate_offer_ratio_robustness(df_unfair, logger)

    return df_unfair


def prepare_hddm_data(
    df: pd.DataFrame, logger: logging.Logger
) -> pd.DataFrame:
    """
    Constructs the canonical data matrix required for HDDM estimation.
    Column schema:
      subj_idx  - Integer subject identifier (HDDM requirement)
      rt        - Reaction time in seconds (HDDM requirement)
      response  - Binary {0, 1} boundary coding (HDDM requirement)
      emotion   - Experimental condition factor
      offer_amount - Offer magnitude (retained for potential covariates)
    """
    df = df.copy()

    unique_ids = df['participant_id'].unique()
    id_map = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}
    df['subj_idx'] = df['participant_id'].map(id_map)

    pd.DataFrame(
        list(id_map.items()),
        columns=['Original_participant_id', 'HDDM_subj_idx']
    ).to_csv('subject_mapping.csv', index=False)

    hddm_df = pd.DataFrame({
        'subj_idx': df['subj_idx'],
        'rt': df['RT'] / 1000.0,
        'response': df['response_hddm'],
        'emotion': df['emotion'],
        'offer_amount': df['Offers_You']
    })

    return hddm_df


# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
def main():
    logger = setup_logger()
    logger.info("=" * 60)
    logger.info("HDDM DATA PREPARATION PIPELINE INITIATED")
    logger.info("=" * 60)

    try:
        input_file = 'trials.csv'
        output_file = 'hddm_data_unfair.csv'

        # Core data processing
        df_unfair = load_and_filter_data(input_file, logger)
        hddm_df = prepare_hddm_data(df_unfair, logger)

        # [FIX-1.1] Trial count adequacy audit
        audit_trial_counts(hddm_df, logger)

        # Export final matrix
        hddm_df.to_csv(output_file, index=False)
        logger.info(f"Final analytical dataset committed to {output_file}")

        # Lineage integration
        generate_data_fingerprint(
            hddm_df, input_file, output_file, logger
        )

        # Verify emotion categories
        emotions_present = hddm_df['emotion'].unique()
        logger.info(f"Emotions preserved for modeling: {emotions_present}")

    except Exception as e:
        logger.error(f"Fatal error encountered: {e}")
        raise


if __name__ == "__main__":
    main()

2026-03-30 09:50:53,685 INFO: ============================================================
2026-03-30 09:50:53,686 INFO: HDDM DATA PREPARATION PIPELINE INITIATED
2026-03-30 09:50:53,687 INFO: ============================================================
2026-03-30 09:50:53,688 INFO: Loading raw data from trials.csv
2026-03-30 09:50:53,736 INFO: Initial raw data dimensions: (16197, 15)
2026-03-30 09:50:53,752 INFO: --- Generating Fair-Ceiling Diagnostics ---
2026-03-30 09:50:53,779 INFO: Fair-ceiling behavior summary exported to 'fair_ceiling_diagnostics.csv'.
2026-03-30 09:50:54,020 INFO: maxp pruned
2026-03-30 09:50:54,030 INFO: cmap pruned
2026-03-30 09:50:54,034 INFO: kern dropped
2026-03-30 09:50:54,035 INFO: post pruned
2026-03-30 09:50:54,037 INFO: FFTM dropped
2026-03-30 09:50:54,041 INFO: GPOS pruned
2026-03-30 09:50:54,045 INFO: GSUB pruned
2026-03-30 09:50:54,046 INFO: name pruned
2026-03-30 09:50:54,054 INFO: glyf pruned
2026-03-30 09:50:54,057 INFO: Added gid0 to subset
2026

2026-03-30 09:50:54,189 INFO: Closed glyph list over 'glyf': 55 glyphs after
2026-03-30 09:50:54,190 INFO: Glyph names: ['.notdef', '.null', 'C', 'E', 'F', 'O', 'P', 'R', 'T', 'U', 'a', 'b', 'c', 'colon', 'comma', 'd', 'e', 'eight', 'f', 'fi', 'five', 'fl', 'four', 'i', 'j', 'l', 'm', 'n', 'nine', 'nonmarkingreturn', 'o', 'one', 'p', 'parenleft', 'parenright', 'percent', 'r', 's', 'six', 'space', 't', 'two', 'u', 'uni239B', 'uni239C', 'uni239D', 'uni239E', 'uni239F', 'uni23A0', 'uniFB00', 'uniFB03', 'uniFB04', 'w', 'y', 'zero']
2026-03-30 09:50:54,191 INFO: Glyph IDs:   [0, 1, 2, 3, 8, 11, 12, 15, 19, 20, 21, 23, 24, 25, 27, 28, 29, 38, 40, 41, 50, 51, 53, 55, 56, 68, 69, 70, 71, 72, 73, 76, 77, 79, 80, 81, 82, 83, 85, 86, 87, 88, 90, 92, 3506, 3507, 3508, 3509, 3510, 3511, 5038, 5039, 5040, 5041, 5042]
2026-03-30 09:50:54,192 INFO: Retaining 55 glyphs
2026-03-30 09:50:54,193 INFO: head subsetting not needed
2026-03-30 09:50:54,194 INFO: hhea subsetting not needed
2026-03-30 09:50:54,1

# Step 2a: Global Configuration

In [2]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 2a: Global Configuration & Cryptographic Lineage (ArviZ-Centric SSOT)
=============================================================================
Pipeline Position: Reads 'data_fingerprint.json' from Step 1.
                   Generates 'hddm_config.py' consumed by Steps 2b-6.

Downstream Consumers: Every subsequent Step imports from hddm_config.py:
  - CFG object (all hyperparameters)
  - load_active_lineage_state() (lineage validation)
  - validate_artifact_lineage() (artifact hash checking)
  - identify_winning_model() (Step 5-6 model routing)
  - load_existing_manifest() (Step 2b checkpoint resume)
  - append_manifest_record() (Step 2b checkpoint persistence)
  - check_memory_headroom() (Step 2b memory safety)

Methodological Basis:
  - Four-parameter DDM (v, a, t, z) without across-trial variability
    (sv, st, sz), per Lerche & Voss (2016) and Boehm et al. (2018)
    recommendations for per-condition trial counts of 30-40.
  - group_only_regressors=True: Treatment contrast coefficients are
    estimated at the group level (fixed effects). Subject-level
    variability is captured by hierarchical DDM base parameters.
    keep_regressor_trace=False: Trial-level regressor traces are
    discarded after sampling to prevent memory exhaustion during
    InferenceData conversion. Group-level posteriors (Intercepts
    and Treatment contrasts) are fully preserved.
    (Wiecki, Sofer & Frank, 2013, Frontiers in Human Neuroscience)
  - Stratified convergence criteria (Vehtari et al., 2021):
    focal parameters require strict ESS, nuisance parameters relaxed.

=============================================================================
"""

import os
import re
import json
import hashlib
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Union


# =============================================================================
# SHARED UTILITY: Lineage Validation (used by all downstream Steps)
# =============================================================================
def load_active_lineage_state(paths: Dict[str, Path], logger=None) -> dict:
    """
    Loads the active pipeline lineage state from the configuration
    fingerprint generated by Step 2a. Returns the dictionary containing
    config_hash, data_hash, and pipeline_hash.

    This function replaces the non-existent 'validate_pipeline_lineage'
    referenced in the original pipeline. All downstream Steps now
    import and call this function consistently.
    """
    fingerprint_path = paths['manifests'] / "config_fingerprint.json"
    if not fingerprint_path.exists():
        raise FileNotFoundError(
            "Missing configuration fingerprint. Execute Step 2a first."
        )

    with open(fingerprint_path, 'r', encoding='utf-8') as f:
        lineage = json.load(f)

    if logger:
        logger.info(
            f"Active Lineage loaded: "
            f"{lineage.get('pipeline_hash', '')[:16]}..."
        )

    return lineage


def validate_artifact_lineage(
    artifact_df, active_lineage: dict, logger=None
) -> str:
    """
    Strictly validates an artifact's lineage against the active
    environment. Checks for the existence of lineage columns and
    enforces hash equality.
    """
    required_fields = ["config_hash", "data_hash", "pipeline_hash"]
    missing = [
        col for col in required_fields
        if col not in artifact_df.columns
    ]

    if missing:
        raise KeyError(
            f"Strict Contract Failure: Artifact missing "
            f"lineage fields {missing}"
        )

    artifact_hash = str(artifact_df['pipeline_hash'].iloc[0])

    if artifact_hash != active_lineage['pipeline_hash']:
        raise RuntimeError(
            f"CRITICAL LINEAGE MISMATCH! Artifact does not belong "
            f"to the current environment.\n"
            f"Artifact Hash: {artifact_hash}\n"
            f"Active Hash: {active_lineage['pipeline_hash']}\n"
            f"Please re-run the pipeline from the out-of-sync step."
        )

    if logger:
        logger.info(
            "Artifact cryptographic lineage validated successfully."
        )

    return artifact_hash


def identify_winning_model(paths: Dict[str, Path]) -> str:
    """
    Parses the Step 4 audit trail to extract the winning model name.
    Strict contract: Demands explicit Is_Winner == True.
    """
    import pandas as pd

    audit_path = paths['audit'] / "final_model_selection_audit.csv"
    if not audit_path.exists():
        raise FileNotFoundError(
            "No audit manifest found. Ensure Step 4 completed."
        )

    df = pd.read_csv(audit_path)
    if df.empty:
        raise RuntimeError("Audit log is empty.")

    if 'Is_Winner' not in df.columns:
        raise KeyError(
            "Strict Contract Failure: 'Is_Winner' column missing "
            "in audit file."
        )

    mask = (
        df["Is_Winner"].astype(str).str.strip().str.lower() == "true"
    )
    winners = df[mask]

    if winners.empty:
        raise ValueError(
            "Strict Contract Failure: No explicit winner flagged."
        )

    if len(winners) > 1:
        raise ValueError(
            "Strict Contract Failure: Multiple winners flagged."
        )

    return str(winners.iloc[0]["model_name"]).lower()


# =============================================================================
# CORE CONFIGURATION DATACLASS
# =============================================================================
@dataclass
class HDDMConfig:
    """
    Analytical configuration schema for the dockerHDDM ArviZ-centric
    pipeline. Manages MCMC hyperparameters, convergence criteria,
    model architecture definitions, and PPC adequacy thresholds.
    """

    # -------------------------------------------------------------------------
    # 1. OPERATIONAL MODE & ROUTING
    # -------------------------------------------------------------------------
    run_mode: str = 'final'    # debug or final
    base_dir: Path = Path(os.getcwd())

    # -------------------------------------------------------------------------
    # 2. STRUCTURAL MODELING CONSTRAINTS
    # -------------------------------------------------------------------------
    # Four-parameter DDM: v, a, t are estimated by default in HDDM;
    # z (starting point bias) requires explicit inclusion via include.
    # sv, st, sz excluded per Lerche & Voss (2016) given 30-40
    # trials per condition.
    include_params: List[str] = field(
        default_factory=lambda: ['z']
    )

    # group_only_regressors=True: Treatment contrast coefficients are
    # estimated as fixed effects at the group level (shared across all
    # subjects). Subject-level variability is captured by the hierarchical
    # structure of the DDM base parameters (v, a, t, z intercepts).
    #
    # Methodological note: group_only_regressors=False would allow
    # per-subject regression betas via partial pooling (Rouder & Lu,
    # 2005), which can yield more honest group-level uncertainty.
    # However, the dockerHDDM implementation (hcp4715/hddm:latest)
    # raises AttributeError in wiener_multi_like when False is used
    # with HDDMRegressor, because the likelihood function expects
    # trial-indexed Series but receives scalar floats for hierarchical
    # beta nodes. This is a known implementation limitation, not a
    # methodological choice.
    #
    # With 30-40 trials/condition, Wiecki et al. (2013, Figure 3) show
    # that hierarchical shrinkage makes group-level conclusions nearly
    # identical between True and False settings.
    group_only_regressors: bool = True

    keep_regressor_trace: bool = False
    p_outlier: float = 0.05
    use_informative_priors: bool = True
    baseline_condition: str = 'neu'

    # -------------------------------------------------------------------------
    # 3. STRATIFIED MCMC DIAGNOSTIC THRESHOLDS (Vehtari et al., 2021)
    # -------------------------------------------------------------------------
    # Focal Parameters (Group-level Intercepts and Treatment contrasts):
    rhat_focal: float = 1.01
    ess_bulk_focal: float = 1000.0
    ess_tail_focal: float = 500.0

    # Nuisance Parameters (Subject-level deviations, transformed scales):
    rhat_nuisance: float = 1.05
    ess_bulk_nuisance: float = 400.0
    ess_tail_nuisance: float = 200.0

    # -------------------------------------------------------------------------
    # 4. EXPERIMENTAL DESIGN & VISUAL STANDARDS
    #    (Excluded from cryptographic hash computation)
    # -------------------------------------------------------------------------
    emotion_order: List[str] = field(
        default_factory=lambda: ['neu', 'rew', 'aff', 'dom', 'dis']
    )

    display_labels: Dict[str, str] = field(
        default_factory=lambda: {
            'neu': 'Neutral', 'rew': 'Reward',
            'aff': 'Affiliative', 'dom': 'Dominance',
            'dis': 'Disgust'
        }
    )

    # Okabe-Ito inspired colorblind-friendly palette
    colors: Dict[str, str] = field(
        default_factory=lambda: {
            'neu': "#8491B4", 'rew': "#3C5488",
            'aff': "#91D1C2", 'dom': "#F39B7F",
            'dis': "#E64B35"
        }
    )

    # -------------------------------------------------------------------------
    # 5. BASE MCMC HYPERPARAMETERS (dockerHDDM ArviZ-centric)
    # -------------------------------------------------------------------------
    # [FIX-2a.2] Unified seed attribute name. All downstream Steps
    # must reference CFG.base_seed (not 'random_seed').
    base_seed: int = 2508

    # [FIX-2a.1] n_chains is initialized to a placeholder here.
    # The actual value is computed in __post_init__ based on
    # self.run_mode, because Python dataclass default values are
    # evaluated at class definition time (not at instantiation),
    # so the conditional expression 'n_chains: int = 2 if
    # run_mode == "debug" else 4' in the original code ALWAYS
    # evaluated to 4 regardless of run_mode.
    n_chains: int = 4  # Placeholder; overridden in __post_init__

    default_thin: int = 2

    # dockerHDDM-specific sampling flags
    enable_loglike: bool = False
    enable_ppc: bool = False
    ppc_samples: int = 500

    # Adaptive sampling limits
    max_adaptive_cycles: int = 5  # Overridden in __post_init__

    # -------------------------------------------------------------------------
    # 6. PPC ADEQUACY THRESHOLDS (Absolute goodness-of-fit)
    # -------------------------------------------------------------------------
    ppc_choice_mae_max: float = 0.05
    ppc_choice_max_err: float = 0.10
    ppc_rt_quantile_mae_max: float = 0.05
    ppc_rt_quantile_max_err: float = 0.10

    # -------------------------------------------------------------------------
    # 7. MODEL ARCHITECTURES & DYNAMIC PROTOCOLS
    # -------------------------------------------------------------------------
    final_core_models: List[str] = field(
        default_factory=lambda: ['null', 'v', 'a', 'va']
    )
    # Exploratory set expanded to ensure each parameter family (z, t)
    # has at least one model where it varies independently alongside v,
    # enabling assessment of its marginal contribution beyond the va
    # baseline. Without vz and vt, z/t could only "ride along" with
    # va in vaz/vazt, conflating their individual contribution with
    # the v+a combination.
    final_exploratory_models: List[str] = field(
        default_factory=lambda: ['vz', 'vt', 'vaz', 'vazt']
    )

    # -------------------------------------------------------------------------
    # 8. MEMORY SAFETY THRESHOLDS
    # -------------------------------------------------------------------------
    min_free_memory_gb: float = 3.0

    # Dynamic fields computed in __post_init__
    mcmc_protocols: Dict[str, Dict[str, Any]] = field(init=False)

    # -------------------------------------------------------------------------
    # DYNAMIC INITIALIZATION
    # -------------------------------------------------------------------------
    def _infer_model_tier(self, model_name: str) -> str:
        """
        Infers computational complexity tier based on the number
        of free DDM parameter families affected by the emotion
        regressor. Tiers determine MCMC sampling budgets.

        Classification logic:
          - null: always simple (intercept-only, no contrasts)
          - 1 family, no hard params (v, a): simple
          - 2 families with hard params (vz, vt): medium
            (z uses logit link, t uses exp link -> more complex
             posterior geometry requiring additional burn-in)
          - 2 families, no hard params (va): simple
          - 3+ families with hard params (vaz, vazt): complex
          - 3+ families, no hard params: medium
        """
        name = model_name.lower()
        if name == 'null':
            return "simple"
        free_families = sum([
            'v' in name, 'a' in name, 'z' in name, 't' in name
        ])
        has_hard_params = ('z' in name) or ('t' in name)

        if (free_families >= 4
                or (free_families >= 3 and has_hard_params)):
            return "complex"
        elif free_families >= 3:
            return "medium"
        elif has_hard_params:
            # Models like vz, vt, az, at: fewer families but
            # nonlinear link functions make posterior geometry
            # harder to explore. Upgrade from simple to medium.
            return "medium"
        else:
            return "simple"

    def __post_init__(self) -> None:
        """
        Computes run-mode-dependent hyperparameters and per-model
        MCMC sampling protocols. This is the ONLY place where
        self.run_mode is used to branch logic.
        """
        # [FIX-2a.1] Correctly set n_chains based on run_mode
        if self.run_mode == 'debug':
            self.n_chains = 2
            self.max_adaptive_cycles = 1
        else:
            self.n_chains = 4
            self.max_adaptive_cycles = 5

        # Build per-model MCMC protocols
        self.mcmc_protocols = {}
        all_models = self.final_core_models + self.final_exploratory_models

        # Tier-specific sampling constraints.
        # group_only_regressors=False increases parameter count
        # substantially, so budgets are set conservatively.
        tiers_config = {
            "simple": {
                "burn": 2000, "target_kept": 3000, "max_kept": 8000
            },
            "medium": {
                "burn": 3000, "target_kept": 4000, "max_kept": 10000
            },
            "complex": {
                "burn": 5000, "target_kept": 5000, "max_kept": 15000
            },
        }

        for model_name in all_models:
            tier = self._infer_model_tier(model_name)
            proto = tiers_config[tier]

            if self.run_mode == 'debug':
                self.mcmc_protocols[model_name] = {
                    "tier": tier,
                    "n_samples": 600,
                    "burn": 100,
                    "thin": 1
                }
            else:
                self.mcmc_protocols[model_name] = {
                    "tier": tier,
                    "n_samples": proto["burn"] + proto["target_kept"] * self.default_thin,
                    "burn": proto["burn"],
                    "thin": self.default_thin,
                    "max_samples": proto["burn"] + proto["max_kept"] * self.default_thin
                }

    # -------------------------------------------------------------------------
    # PARAMETER CLASSIFICATION
    # -------------------------------------------------------------------------
    @staticmethod
    def identify_focal_parameters(param_names: List[str]) -> List[str]:
        """
        Separates focal inferential parameters (group-level Intercepts
        and Treatment contrasts) from hierarchical nuisance parameters
        (subject-level deviations, transformed scales, variance terms).
        """
        nuisance_pattern = re.compile(
            r"(_subj|_trans|_std|_var|_log|\.[\d]+$)"
        )
        return [
            p for p in param_names
            if not nuisance_pattern.search(p)
        ]

    # -------------------------------------------------------------------------
    # COMPUTED PROPERTIES
    # -------------------------------------------------------------------------
    @property
    def final_all_models(self) -> List[str]:
        """Aggregation of core and exploratory model architectures."""
        return self.final_core_models + self.final_exploratory_models

    # -------------------------------------------------------------------------
    # DIRECTORY MANAGEMENT
    # -------------------------------------------------------------------------
    def initialize_directories(self) -> Dict[str, Path]:
        """Constructs and validates the publication output directory tree."""
        subdirs = [
            'manifests', 'audit', 'models', 'ppc', 'recovery',
            'figures_main', 'figures_supp', 'tables_main', 'tables_supp'
        ]
        root_out = self.base_dir / f"results_hddm_{self.run_mode}"
        paths = {name: root_out / name for name in subdirs}
        for p in paths.values():
            p.mkdir(parents=True, exist_ok=True)
        return paths

    # -------------------------------------------------------------------------
    # CRYPTOGRAPHIC FINGERPRINT -> PIPELINE LINEAGE FINGERPRINT
    # -------------------------------------------------------------------------
    def generate_pipeline_fingerprint(self) -> Dict[str, Any]:
        """
        Computes deterministic SHA-256 hash of structural hyperparameters,
        incorporates the empirical data hash, and generates a unified
        pipeline hash. Aesthetic parameters (colors, labels) are
        explicitly excluded from the hash computation.
        """
        critical_keys = [
            'run_mode', 'include_params',
            'group_only_regressors', 'p_outlier',
            'use_informative_priors', 'baseline_condition',
            'n_chains', 'base_seed',
            'rhat_focal', 'ess_bulk_focal', 'ess_tail_focal',
            'rhat_nuisance', 'ess_bulk_nuisance',
            'ess_tail_nuisance',
            'default_thin', 'enable_loglike', 'enable_ppc',
            'ppc_choice_mae_max', 'ppc_rt_quantile_mae_max',
            'final_core_models', 'final_exploratory_models',
            'mcmc_protocols'
        ]

        cfg_dict = asdict(self)
        structural_params = {k: cfg_dict[k] for k in critical_keys}

        # 1. Configuration Hash
        json_str = json.dumps(structural_params, sort_keys=True)
        config_hash = hashlib.sha256(
            json_str.encode('utf-8')
        ).hexdigest()

        # 2. Data Hash (Acquired from Step 1 Artifact)
        data_fingerprint_path = self.base_dir / "data_fingerprint.json"
        data_hash = ""
        if data_fingerprint_path.exists():
            with open(data_fingerprint_path, 'r', encoding='utf-8') as f:
                data_hash = json.load(f).get('data_file_hash', '')
        else:
            print(
                "WARNING: 'data_fingerprint.json' not found. "
                "Data lineage will be empty."
            )

        # 3. Holistic Pipeline Hash
        pipeline_string = f"{config_hash}_{data_hash}"
        pipeline_hash = hashlib.sha256(
            pipeline_string.encode('utf-8')
        ).hexdigest()

        return {
            'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
            'config_hash': config_hash,
            'data_hash': data_hash,
            'pipeline_hash': pipeline_hash,
            'structural_parameters': structural_params
        }


# =============================================================================
# [FIX-2a.7] ROBUST CONFIG SERIALIZATION VIA JSON
# =============================================================================
def _serialize_config_to_disk(
    cfg_obj: HDDMConfig, target_paths: Dict[str, Path]
) -> dict:
    """
    Serializes the configuration state to a Python module
    ('hddm_config.py') for cross-session recovery after kernel
    restarts.

    [FIX-2a.7] Rewritten to use JSON-based value persistence instead
    of line-by-line f.write(). The generated module loads a JSON file
    and reconstructs the config object at import time, eliminating
    syntax errors from special characters in paths.

    [FIX-2a.3] Writes load_active_lineage_state() (replacing the
    non-existent validate_pipeline_lineage), plus all other utility
    functions needed by downstream Steps.

    [FIX-2a.5] Writes check_memory_headroom() for Step 2b safety.

    [FIX-2a.6] Writes load_existing_manifest() and
    append_manifest_record() for Step 2b checkpoint resume.
    """
    # --- Phase 1: Persist config values as JSON ---
    cfg_values_path = 'hddm_config_values.json'
    cfg_dict = asdict(cfg_obj)

    # Convert Path objects to strings for JSON serialization
    serializable = {}
    for k, v in cfg_dict.items():
        if isinstance(v, Path):
            serializable[k] = str(v)
        else:
            serializable[k] = v

    with open(cfg_values_path, 'w', encoding='utf-8') as f:
        json.dump(serializable, f, indent=2, default=str)

    # --- Phase 2: Generate hddm_config.py module ---
    py_config_path = 'hddm_config.py'
    module_code = '''# -*- coding: utf-8 -*-
"""
Auto-generated SSOT configuration module.
DO NOT EDIT MANUALLY. Regenerate by running Step 2a.
"""
import os
import re
import gc
import json
import pandas as pd
from pathlib import Path


# =============================================================================
# CONFIGURATION LOADER
# =============================================================================
def _load_config_from_json():
    """
    Reconstructs configuration state from the JSON persistence file
    generated by Step 2a serialization.
    """
    config_json_path = Path(__file__).parent / "hddm_config_values.json"
    if not config_json_path.exists():
        raise FileNotFoundError(
            f"Configuration JSON not found at {config_json_path}. "
            f"Re-execute Step 2a."
        )
    with open(config_json_path, 'r', encoding='utf-8') as f:
        return json.load(f)


class _HDDMConfig:
    """Runtime configuration object reconstructed from JSON persistence."""

    def __init__(self):
        vals = _load_config_from_json()
        for k, v in vals.items():
            if k == 'base_dir':
                setattr(self, k, Path(v))
            else:
                setattr(self, k, v)

    @property
    def final_all_models(self):
        return self.final_core_models + self.final_exploratory_models

    def initialize_directories(self):
        subdirs = [
            'manifests', 'audit', 'models', 'ppc', 'recovery',
            'figures_main', 'figures_supp', 'tables_main',
            'tables_supp'
        ]
        root = self.base_dir / f"results_hddm_{self.run_mode}"
        paths = {n: root / n for n in subdirs}
        for p in paths.values():
            p.mkdir(parents=True, exist_ok=True)
        return paths

    @staticmethod
    def identify_focal_parameters(param_names):
        pat = re.compile(r"(_subj|_trans|_std|_var|_log|\\.\\d+$)")
        return [p for p in param_names if not pat.search(p)]


# Instantiate singleton configuration object
CFG = _HDDMConfig()


# =============================================================================
# LINEAGE VALIDATION UTILITIES
# =============================================================================
def load_active_lineage_state(paths, logger=None):
    """
    Loads the active pipeline lineage state from the configuration
    fingerprint. Returns dict with config_hash, data_hash,
    pipeline_hash.

    This is the canonical lineage validation entry point for ALL
    downstream Steps (2b, 3, 4, 5, 6).
    """
    fp = paths['manifests'] / 'config_fingerprint.json'
    if not fp.exists():
        raise FileNotFoundError(
            'Missing lineage fingerprint. Execute Step 2a.'
        )
    with open(fp, 'r', encoding='utf-8') as fh:
        lineage = json.load(fh)
    if logger:
        logger.info(
            f"Active Pipeline Hash: "
            f"{lineage.get('pipeline_hash', '')[:16]}..."
        )
    return lineage


def validate_artifact_lineage(artifact_df, active_lineage, logger=None):
    """
    Validates an artifact DataFrame's lineage hashes against
    the active pipeline configuration.
    """
    required_fields = ['config_hash', 'data_hash', 'pipeline_hash']
    missing = [
        col for col in required_fields
        if col not in artifact_df.columns
    ]
    if missing:
        raise KeyError(
            f'Strict Contract Failure: Artifact missing '
            f'lineage fields {missing}'
        )
    artifact_hash = str(artifact_df['pipeline_hash'].iloc[0])
    if artifact_hash != active_lineage['pipeline_hash']:
        raise RuntimeError(
            f'CRITICAL LINEAGE MISMATCH!\\n'
            f'Artifact Hash: {artifact_hash}\\n'
            f'Active Hash: {active_lineage["pipeline_hash"]}'
        )
    if logger:
        logger.info(
            'Artifact cryptographic lineage validated successfully.'
        )
    return artifact_hash


def identify_winning_model(paths):
    """
    Parses the Step 4 audit trail to extract the winning model name.
    Strict contract: exactly one model must have Is_Winner == True.
    """
    audit_path = paths['audit'] / 'final_model_selection_audit.csv'
    if not audit_path.exists():
        raise FileNotFoundError(
            'No audit manifest found. '
            'Ensure Step 4 completed successfully.'
        )
    df = pd.read_csv(audit_path)
    if 'Is_Winner' not in df.columns:
        raise KeyError(
            "Strict Contract Failure: "
            "'Is_Winner' column missing in audit file."
        )
    mask = (
        df['Is_Winner'].astype(str).str.strip().str.lower() == 'true'
    )
    winners = df[mask]
    if winners.empty:
        raise ValueError(
            'Strict Contract Failure: No explicit winner flagged.'
        )
    if len(winners) > 1:
        raise ValueError(
            'Strict Contract Failure: '
            'Multiple winners flagged ambiguously.'
        )
    return str(winners.iloc[0]['model_name']).lower()


# =============================================================================
# [FIX-2a.6] MODEL-LEVEL CHECKPOINT UTILITIES (for Step 2b)
# =============================================================================
def load_existing_manifest(manifest_path):
    """
    Loads previously completed model records from disk for
    checkpoint resume. Returns a tuple of:
      - set of completed model names (to skip)
      - list of existing manifest record dicts (to preserve)

    A model is considered 'completed' if it has a manifest entry
    without an 'error' field, AND its .nc file exists on disk.
    """
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        return set(), []

    df = pd.read_csv(manifest_path)
    if df.empty:
        return set(), []

    # Only count models that completed without error
    if 'error' in df.columns:
        completed_mask = df['error'].isna()
    else:
        completed_mask = pd.Series([True] * len(df))

    completed_names = set(df.loc[completed_mask, 'model_name'].tolist())

    return completed_names, df.to_dict('records')


def append_manifest_record(manifest_path, record):
    """
    Atomically appends a single model record to the manifest CSV.
    Uses write-to-temp-then-rename pattern to prevent data loss
    from partial writes during power failure.

    On POSIX systems, os.replace() is atomic (IEEE Std 1003.1).
    On Windows/Docker-on-Windows, this provides best-effort
    protection.
    """
    manifest_path = Path(manifest_path)
    tmp_path = manifest_path.with_suffix('.csv.tmp')

    if manifest_path.exists():
        df_existing = pd.read_csv(manifest_path)
        df_new = pd.concat(
            [df_existing, pd.DataFrame([record])],
            ignore_index=True
        )
    else:
        df_new = pd.DataFrame([record])

    df_new.to_csv(tmp_path, index=False)

    # Atomic replacement
    if os.name == 'nt':
        # Windows: os.replace is atomic on NTFS
        os.replace(str(tmp_path), str(manifest_path))
    else:
        # POSIX: os.replace is guaranteed atomic
        os.replace(str(tmp_path), str(manifest_path))


# =============================================================================
# [FIX-2a.5] MEMORY SAFETY GUARD (for Step 2b)
# =============================================================================
def check_memory_headroom(min_free_gb=None, logger=None):
    """
    Verifies sufficient free memory before initiating model estimation.
    Prevents OOM-induced system hang in Docker containers configured
    with --oom-kill-disable.

    The default threshold (from CFG.min_free_memory_gb) accounts for:
      - Peak PyMC2 runtime overhead (~2GB for 4 parallel chains)
      - Log-likelihood matrix assembly (~1GB for typical datasets)
      - Safety margin for OS/kernel buffers
    """
    try:
        import psutil
    except ImportError:
        if logger:
            logger.warning(
                "psutil not available. Memory check skipped."
            )
        return True

    if min_free_gb is None:
        min_free_gb = getattr(CFG, 'min_free_memory_gb', 3.0)

    mem = psutil.virtual_memory()
    free_gb = mem.available / (1024 ** 3)

    if free_gb < min_free_gb:
        if logger:
            logger.warning(
                f"Memory headroom low: {free_gb:.1f}GB free "
                f"(minimum {min_free_gb}GB). Forcing gc..."
            )
        gc.collect()

        mem = psutil.virtual_memory()
        free_gb = mem.available / (1024 ** 3)

        if free_gb < min_free_gb:
            raise MemoryError(
                f"CRITICAL: Only {free_gb:.1f}GB available after gc. "
                f"Cannot safely initiate MCMC sampling. "
                f"Consider reducing n_chains or restarting kernel."
            )

    if logger:
        logger.info(f"Memory check passed: {free_gb:.1f}GB available.")
    return True
'''

    with open(py_config_path, 'w', encoding='utf-8') as f:
        f.write(module_code)

    # --- Phase 3: Persist cryptographic fingerprint ---
    lineage_state = cfg_obj.generate_pipeline_fingerprint()
    fp_path = target_paths['manifests'] / "config_fingerprint.json"
    with open(fp_path, 'w', encoding='utf-8') as f:
        json.dump(lineage_state, f, indent=2, default=str)

    return lineage_state


# =============================================================================
# EXECUTION & PERSISTENCE
# =============================================================================

# Instantiate configuration in active kernel memory
CFG = HDDMConfig()
PATHS = CFG.initialize_directories()

# Execute serialization
active_lineage = _serialize_config_to_disk(CFG, PATHS)

# Print confirmation
print("=" * 70)
print(f"  STEP 2a: Configuration SSOT & Pipeline Lineage Initialized")
print(f"  Run Mode:              {CFG.run_mode}")
print(f"  DDM Parameters:        v, a, t + {CFG.include_params}")
print(f"  Chains:                {CFG.n_chains}")
print(f"  Base Seed:             {CFG.base_seed}")
print(f"  group_only_regressors: {CFG.group_only_regressors}")
print(f"  Thin:                  {CFG.default_thin}")
print(f"  keep_regressor_trace:  {CFG.keep_regressor_trace}")
print(f"  LogLike (WAIC):        {CFG.enable_loglike}")
print(f"  Models:                {len(CFG.final_all_models)} architectures")
print("-" * 70)
print(f"  Config Hash:           {active_lineage['config_hash'][:16]}...")
print(f"  Data Hash:             {active_lineage['data_hash'][:16]}...")
print(f"  PIPELINE HASH:         {active_lineage['pipeline_hash'][:16]}...")
print("=" * 70)

# Display per-model protocols
for model_name in CFG.final_all_models:
    proto = CFG.mcmc_protocols[model_name]
    print(
        f"  {model_name:>6s} | tier={proto['tier']:>7s} | "
        f"samples={proto['n_samples']:>5d} | burn={proto['burn']:>4d} | thin={proto['thin']}"
    )

  STEP 2a: Configuration SSOT & Pipeline Lineage Initialized
  Run Mode:              final
  DDM Parameters:        v, a, t + ['z']
  Chains:                4
  Base Seed:             2508
  group_only_regressors: True
  Thin:                  2
  keep_regressor_trace:  False
  LogLike (WAIC):        False
  Models:                8 architectures
----------------------------------------------------------------------
  Config Hash:           5cce0d3c92ade507...
  Data Hash:             a80b1f0721bdf6f4...
  PIPELINE HASH:         c801cab37a7dfbe3...
    null | tier= simple | samples= 8000 | burn=2000 | thin=2
       v | tier= simple | samples= 8000 | burn=2000 | thin=2
       a | tier= simple | samples= 8000 | burn=2000 | thin=2
      va | tier= simple | samples= 8000 | burn=2000 | thin=2
      vz | tier= medium | samples=11000 | burn=3000 | thin=2
      vt | tier= medium | samples=11000 | burn=3000 | thin=2
     vaz | tier=complex | samples=15000 | burn=5000 | thin=2
    vazt | tier=c

# Step 2b: Hierarchical Bayesian Model Specification and MCMC Estimation

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 2b: Hierarchical Bayesian Model Specification & Adaptive MCMC
         Estimation (dockerHDDM ArviZ-Centric Workflow)
=============================================================================
Pipeline Position:
  Upstream:   Step 2a (imports CFG and utilities from hddm_config.py;
              reads 'hddm_data_unfair.csv' from Step 1)
  Downstream: Step 3 (reads .nc and .hddm files from models/ directory;
              reads model_manifest.csv from manifests/ directory)

Methodological Purpose:
  - Executes parallel-chain MCMC sampling via dockerHDDM for each model
    architecture defined in the configuration SSOT.
  - Generates ArviZ InferenceData objects (.nc) and serialized HDDM
    model objects (.hddm) for downstream consumption.
  - Implements a Dynamic Regressor Factory to construct Patsy formulas
    for treatment-coded condition effects.
  - Enforces continuous MCMC convergence evaluation using stratified
    diagnostics (focal vs. nuisance parameters) via adaptive sampling.
  - Implements model-level checkpoint resume: completed models are
    skipped on pipeline restart after interruption.
  - Implements memory safety guard before each model estimation.

Statistical Assumptions & Parameters:
  - Assumes input data contains positive reaction times (RT) in seconds.
  - Convergence criteria utilize Gelman-Rubin R-hat and Effective Sample
    Size (ESS) metrics (Vehtari et al., 2021; Gelman et al., 2020).
  - group_only_regressors=True: Treatment contrast betas are estimated
    at the group level. keep_regressor_trace=False: trial-level
    regressor traces discarded to prevent OOM during InferenceData
    conversion (Wiecki et al., 2013; Pan et al., 2025).

=============================================================================
"""

import os
import gc
import glob
import time
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import arviz as az
import hddm

# -----------------------------------------------------------------------------
# [FIX-2b.1] CONFIGURATION IMPORT
# Import load_active_lineage_state (replaces validate_pipeline_lineage)
# Import checkpoint utilities and memory guard from Step 2a serialized module
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state,
        load_existing_manifest,
        append_manifest_record,
        check_memory_headroom
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execution of Step 2a is mandatory to generate "
        "configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# [FIX-2b.2] Establish deterministic behavior using the configured seed
np.random.seed(CFG.base_seed)


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_estimation_logger() -> logging.Logger:
    """
    Initializes dual-sink logging infrastructure (File + Stream)
    for MCMC estimation tracking. Timestamp-stamped log file prevents
    overwrites across multiple pipeline runs.
    """
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_mcmc_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'mcmc_estimation_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# [FIX-2b.3] DYNAMIC REGRESSOR FACTORY
# =============================================================================
def build_model_regressors(
    model_name: str, baseline: str
) -> List[str]:
    """
    Constructs Patsy formulas for hddm.HDDMRegressor.

    CRITICAL dockerHDDM COMPATIBILITY PATTERN:
    Every DDM parameter (v, a, t, z) MUST have an explicit regression
    formula — either treatment-coded ("v ~ C(emotion, ...)") if the
    parameter varies by condition, or intercept-only ("v ~ 1") if it
    does not. This is because dockerHDDM's wiener_multi_like expects
    ALL parameters to be created as regression nodes (producing
    trial-indexed Pandas Series). Parameters without a formula are
    handled by HDDM's default node constructor, which produces scalar
    floats, causing AttributeError on .loc access.

    This pattern was validated on hcp4715/hddm:latest and matches
    the standard HDDMRegressor usage documented in Pan et al. (2022).

    The null model also uses this pattern with ALL parameters as
    intercept-only. This ensures ALL models share identical
    HDDMRegressor parameterization, making DIC/LOO/WAIC comparisons
    valid across the entire model space.

    Parameters:
      model_name: Architecture specification string.
      baseline:   Reference condition level for Treatment coding.

    Returns:
      List of 4 Patsy formula strings (one per DDM parameter).
    """
    name_lower = model_name.lower()
    regressors = []

    for param in ['v', 'a', 't', 'z']:
        if name_lower != 'null' and param in name_lower:
            # This parameter varies by emotion condition
            regressors.append(
                f"{param} ~ C(emotion, Treatment('{baseline}'))"
            )
        else:
            # This parameter is intercept-only (constant across conditions)
            regressors.append(f"{param} ~ 1")

    return regressors


# =============================================================================
# EMPIRICAL DATA VALIDATION
# =============================================================================
def validate_empirical_data(
    data_path: str, logger: logging.Logger
) -> pd.DataFrame:
    """
    Enforces HDDM structural requirements and datatype constraints
    on the empirical design matrix prior to estimation.

    Validates:
      - Mandatory columns: subj_idx, rt, response
      - RT values are numeric and positive
      - Response values are binary {0, 1}
      - Baseline emotion condition exists in the data
      - No NaN values in critical columns
    """
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"Input data path unresolved: {data_path}"
        )

    df = pd.read_csv(data_path)
    logger.info(
        f"Empirical data loaded: {df.shape[0]} trials, "
        f"{df.shape[1]} columns"
    )

    # HDDM structural dependency validation
    required_cols = {'subj_idx', 'rt', 'response'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Mandatory HDDM columns missing: {missing}"
        )

    # Datatype normalization
    df['subj_idx'] = df['subj_idx'].astype(str)
    df['rt'] = pd.to_numeric(df['rt'], errors='coerce')
    df['response'] = pd.to_numeric(df['response'], errors='coerce')

    # Data integrity enforcement: exclude NaN entries
    n_before = len(df)
    df = df.dropna(subset=['rt', 'response'])
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        logger.warning(
            f"Data exclusion: {n_dropped} rows dropped "
            f"due to NaN rt/response."
        )

    # Baseline condition verification
    if 'emotion' in df.columns:
        if CFG.baseline_condition not in df['emotion'].unique():
            raise ValueError(
                f"Baseline condition '{CFG.baseline_condition}' "
                f"absent. Available levels: "
                f"{df['emotion'].unique().tolist()}"
            )

    n_subjects = df['subj_idx'].nunique()
    n_conditions = (
        df['emotion'].nunique() if 'emotion' in df.columns else 0
    )
    logger.info(
        f"Validation complete: {n_subjects} subjects, "
        f"{n_conditions} conditions, {len(df)} trials retained."
    )
    logger.info(
        f"RT bounds: [{df['rt'].min():.3f}, "
        f"{df['rt'].max():.3f}] seconds."
    )

    return df


# =============================================================================
# STRATIFIED CONVERGENCE DIAGNOSTICS
# =============================================================================
def evaluate_stratified_convergence(
    infdata: az.InferenceData, logger: logging.Logger
) -> Tuple[bool, Dict[str, float], pd.DataFrame]:
    """
    Computes Gelman-Rubin (R-hat) and Effective Sample Size (ESS)
    statistics. Applies stratified thresholds based on parameter
    classification (focal vs. nuisance).

    Focal parameters (group-level intercepts and treatment contrasts)
    require stricter thresholds because they are the primary
    inferential targets. Nuisance parameters (subject-level
    deviations, variance terms) use relaxed thresholds because
    hierarchical shrinkage makes them inherently less variable.

    Reference: Vehtari et al. (2021), "Rank-normalization,
    folding, and localization", Bayesian Analysis.

    Parameters:
      infdata: ArviZ InferenceData with posterior group.
      logger:  Active logging instance.

    Returns:
      converged:  Boolean indicating all criteria satisfied.
      metrics:    Dictionary of computed diagnostic extrema.
      summary_df: Full ArviZ statistical summary DataFrame.
    """
    summary_df = az.summary(infdata, round_to=4, hdi_prob=0.95)

    # Parameter stratification
    focal_params = CFG.identify_focal_parameters(
        summary_df.index.tolist()
    )
    summary_df['param_class'] = [
        'focal' if p in focal_params else 'nuisance'
        for p in summary_df.index
    ]

    focal_df = summary_df[summary_df["param_class"] == "focal"]
    nuisance_df = summary_df[summary_df["param_class"] == "nuisance"]

    # Extremum metric extraction
    metrics = {
        "f_rhat": (
            focal_df["r_hat"].max()
            if not focal_df.empty else 1.0
        ),
        "f_bulk": (
            focal_df["ess_bulk"].min()
            if not focal_df.empty else float("inf")
        ),
        "f_tail": (
            focal_df["ess_tail"].min()
            if ("ess_tail" in focal_df.columns
                and not focal_df.empty)
            else float("inf")
        ),
        "n_rhat": (
            nuisance_df["r_hat"].max()
            if not nuisance_df.empty else 1.0
        ),
        "n_bulk": (
            nuisance_df["ess_bulk"].min()
            if not nuisance_df.empty else float("inf")
        ),
        "n_tail": (
            nuisance_df["ess_tail"].min()
            if ("ess_tail" in nuisance_df.columns
                and not nuisance_df.empty)
            else float("inf")
        ),
    }

    # Criteria evaluation
    focal_ok = (
        (metrics["f_rhat"] <= CFG.rhat_focal)
        and (metrics["f_bulk"] >= CFG.ess_bulk_focal)
        and (metrics["f_tail"] >= CFG.ess_tail_focal)
    )

    nuisance_ok = (
        (metrics["n_rhat"] <= CFG.rhat_nuisance)
        and (metrics["n_bulk"] >= CFG.ess_bulk_nuisance)
        and (metrics["n_tail"] >= CFG.ess_tail_nuisance)
    )

    converged = focal_ok and nuisance_ok

    logger.info(
        f"  FOCAL   | R-hat={metrics['f_rhat']:.3f} "
        f"(<={CFG.rhat_focal}) | "
        f"ESS_bulk={metrics['f_bulk']:.0f} "
        f"(>={CFG.ess_bulk_focal}) | "
        f"ESS_tail={metrics['f_tail']:.0f} "
        f"(>={CFG.ess_tail_focal}) | "
        f"{'PASS' if focal_ok else 'FAIL'}"
    )
    logger.info(
        f"  NUISANCE| R-hat={metrics['n_rhat']:.3f} "
        f"(<={CFG.rhat_nuisance}) | "
        f"ESS_bulk={metrics['n_bulk']:.0f} "
        f"(>={CFG.ess_bulk_nuisance}) | "
        f"ESS_tail={metrics['n_tail']:.0f} "
        f"(>={CFG.ess_tail_nuisance}) | "
        f"{'PASS' if nuisance_ok else 'FAIL'}"
    )

    return converged, metrics, summary_df


# =============================================================================
# CORE: SINGLE MODEL FITTING PIPELINE
# =============================================================================
def fit_single_model(
    model_name: str,
    data: pd.DataFrame,
    logger: logging.Logger,
    config_hash: str
) -> Dict[str, Any]:
    """
    Executes parameter estimation for a specified model architecture.

    Workflow:
      Phase 1: Construct model architecture (HDDMRegressor for ALL
               models including null).
      Phase 2: Execute initial MCMC sampling with dockerHDDM native
               parallel chains.
      Phase 3: Evaluate convergence; if insufficient, run adaptive
               sampling cycles with ESS-deficit-proportional extensions.
      Phase 4: Export diagnostics and return manifest record.

    Parameters:
      model_name:  Architecture specification string.
      data:        Validated empirical matrix.
      logger:      Active logging instance.
      config_hash: Active SHA-256 fingerprint for lineage tagging.

    Returns:
      Manifest dictionary with convergence status, file paths,
      and diagnostic metrics.
    """
    logger.info(f"\n{'='*70}")
    logger.info(
        f"ESTIMATION SEQUENCE INITIATED: [{model_name.upper()}]"
    )
    logger.info(f"{'='*70}")

    proto = CFG.mcmc_protocols[model_name]
    logger.info(
        f"  Protocol: tier={proto['tier']}, "
        f"n_samples={proto['n_samples']}, burn={proto['burn']}, "
        f"chains={CFG.n_chains}"
    )

    # -----------------------------------------------------------------
    # [FIX-2b.5] MEMORY SAFETY CHECK
    # -----------------------------------------------------------------
    check_memory_headroom(logger=logger)

    # -----------------------------------------------------------------
    # PHASE 1: Architecture Construction
    # ALL models (including null) use HDDMRegressor with explicit
    # formulas for ALL 4 DDM parameters. This is the ONLY pattern
    # confirmed to work on dockerHDDM (hcp4715/hddm:latest).
    #
    # Every parameter gets either:
    #   - Treatment-coded formula (varies by condition)
    #   - Intercept-only "~ 1" formula (constant across conditions)
    #
    # include=['v','a','t','z'] ensures all 4 parameters are
    # created as regression nodes (producing trial-indexed Series),
    # preventing the scalar/.loc AttributeError in wiener_multi_like.
    # -----------------------------------------------------------------
    regressors = build_model_regressors(
        model_name, CFG.baseline_condition
    )

    # All 4 DDM parameters explicitly included
    full_include = ['v', 'a', 't', 'z']

    model = hddm.HDDMRegressor(
        data,
        regressors,
        include=full_include,
        is_group_model=True,
        group_only_regressors=CFG.group_only_regressors,
        keep_regressor_trace=CFG.keep_regressor_trace,
        informative=CFG.use_informative_priors,
        p_outlier=CFG.p_outlier
    )
    logger.info(f"  Architecture: hddm.HDDMRegressor")
    logger.info(f"  include={full_include}")
    for reg in regressors:
        logger.info(f"    -> {reg}")

    # -----------------------------------------------------------------
    # PHASE 2: Initial Posterior Sampling
    # [FIX-2b.7] Native parallel chains via dockerHDDM
    # -----------------------------------------------------------------
    save_prefix = str(PATHS['models'] / f"hddm_{model_name}")
    enable_loglike = CFG.enable_loglike

    t_start = time.time()
    logger.info(
        f"  Sampling: {proto['n_samples']} iterations x "
        f"{CFG.n_chains} chains (burn={proto['burn']})..."
    )

    try:
        infdata = model.sample(
            proto['n_samples'],
            burn=proto['burn'],
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=enable_loglike,
            ppc=CFG.enable_ppc,
            n_ppc=CFG.ppc_samples if CFG.enable_ppc else None,
            save_name=save_prefix
        )
    except MemoryError:
        # Fallback: bypass pointwise log-likelihood matrix.
        # This restricts downstream comparison to DIC only;
        # WAIC/LOO-CV become unavailable for this model.
        logger.warning(
            "  MemoryError with loglike=True. "
            "Fallback: sampling without log-likelihood."
        )
        enable_loglike = False
        gc.collect()

        infdata = model.sample(
            proto['n_samples'],
            burn=proto['burn'],
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=False,
            ppc=CFG.enable_ppc,
            n_ppc=CFG.ppc_samples if CFG.enable_ppc else None,
            save_name=save_prefix
        )

    t_elapsed = time.time() - t_start
    logger.info(
        f"  Initial sampling completed in {t_elapsed/60:.1f} minutes."
    )

    # -----------------------------------------------------------------
    # PHASE 3: Adaptive Convergence Loop
    # -----------------------------------------------------------------
    converged, metrics, summary_df = evaluate_stratified_convergence(
        infdata, logger
    )

    total_samples = proto['n_samples']
    max_samples = proto.get(
        'max_samples', proto['n_samples'] * 2
    )
    cycle = 0

    while not converged and cycle < CFG.max_adaptive_cycles:
        cycle += 1

        # Compute ESS-deficit-proportional extension size
        target_ess = CFG.ess_bulk_focal
        current_min_ess = max(metrics["f_bulk"], 1.0)
        deficit_ratio = target_ess / current_min_ess
        added_samples = max(
            1000, int(total_samples * (deficit_ratio - 1) * 1.25)
        )

        if total_samples + added_samples > max_samples:
            added_samples = max_samples - total_samples
            if added_samples <= 0:
                logger.warning(
                    f"  Sample ceiling ({max_samples}) reached. "
                    f"Terminating adaptive cycles."
                )
                break

        logger.info(
            f"\n  [Adaptive Cycle {cycle}/{CFG.max_adaptive_cycles}] "
            f"Appending {added_samples} samples "
            f"(Deficit Ratio: {deficit_ratio:.2f})..."
        )

        model.sample(
            added_samples,
            burn=0,
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=False,
            ppc=False,
            save_name=save_prefix
        )

        total_samples += added_samples

        # [FIX-2b.6] Regenerate InferenceData from model object
        # CRITICAL: Cannot rely on .nc file because initial save
        # may have failed (e.g., 'index' error or file lock).
        # model.to_infdata() rebuilds InferenceData directly from
        # the live PyMC2 trace, which always reflects appended samples.
        try:
            infdata = model.to_infdata(
                loglike=False, ppc=False
            )
            logger.info(
                f"  InferenceData refreshed from model object."
            )
        except Exception as e:
            logger.warning(
                f"  model.to_infdata() failed: {e}. "
                f"Diagnostics may reflect stale trace."
            )

        converged, metrics, summary_df = (
            evaluate_stratified_convergence(infdata, logger)
        )

    # -----------------------------------------------------------------
    # PHASE 4: Explicit Artifact Persistence, Diagnostics & Cleanup
    #
    # CRITICAL: dockerHDDM's save_name parameter only generates .db
    # trace files (PyMC2 database), NOT .hddm or .nc files. We must
    # explicitly call model.save() and infdata.to_netcdf() to persist
    # the artifacts that downstream Steps 3-5 depend on.
    # -----------------------------------------------------------------

    # 4a. Save serialized HDDM model object (.hddm)
    # Required by: Step 3 (PPC via post_pred_gen), Step 4 (DIC)
    hddm_save_path = f"{save_prefix}.hddm"
    try:
        model.save(hddm_save_path)
        logger.info(
            f"  Model object saved: {Path(hddm_save_path).name}"
        )
    except Exception as e:
        logger.warning(f"  Failed to save .hddm: {e}")

    # 4b. Clean up .db trace files BEFORE .nc save
    # dockerHDDM's save_name creates per-chain .db files that hold
    # file locks. These must be removed before writing .nc to the
    # same prefix path, otherwise "unable to truncate file" error.
    db_pattern = f"{save_prefix}*.db"
    db_files = glob.glob(db_pattern)
    if db_files:
        total_db_mb = sum(
            os.path.getsize(f) for f in db_files
        ) / (1024 * 1024)
        for db_file in db_files:
            try:
                os.remove(db_file)
            except OSError:
                pass
        logger.info(
            f"  Cleaned up {len(db_files)} .db files "
            f"({total_db_mb:.0f} MB freed)"
        )

    # 4c. Save ArviZ InferenceData (.nc)
    # Required by: Step 3 (trace/rank plots), Step 4 (LOO/WAIC),
    #              Step 5 (posterior extraction)
    #
    # The in-memory infdata from model.sample() may have inconsistent
    # dimensions after adaptive sampling cycles. If saving fails,
    # regenerate a clean InferenceData from the model using
    # model.to_infdata() as fallback.
    nc_save_path = f"{save_prefix}.nc"
    nc_saved = False

    # Primary: save the in-memory infdata
    if infdata is not None:
        try:
            infdata.to_netcdf(nc_save_path)
            logger.info(
                f"  InferenceData saved: {Path(nc_save_path).name}"
            )
            nc_saved = True
        except Exception as e:
            logger.warning(
                f"  In-memory infdata save failed: {e}. "
                f"Attempting regeneration via model.to_infdata()..."
            )

    # Fallback: regenerate from model object
    if not nc_saved:
        try:
            infdata_fresh = model.to_infdata(
                loglike=False, ppc=False
            )
            infdata_fresh.to_netcdf(nc_save_path)
            logger.info(
                f"  InferenceData regenerated and saved: "
                f"{Path(nc_save_path).name}"
            )
            # Update infdata reference for downstream use
            infdata = infdata_fresh
            nc_saved = True
        except Exception as e2:
            logger.warning(
                f"  Fallback .nc save also failed: {e2}. "
                f"Step 3 trace plots will be unavailable "
                f"for [{model_name}]. PPC via .hddm still works."
            )

    # 4d. Statistical summary export
    summary_path = PATHS['audit'] / f"summary_{model_name}.csv"
    summary_df.to_csv(summary_path)
    logger.info(
        f"  Statistical summary exported: {summary_path.name}"
    )

    try:
        dic_value = model.dic
    except Exception:
        dic_value = float('inf')
    logger.info(
        f"  Deviance Information Criterion (DIC) = {dic_value:.2f}"
    )

    manifest_record = {
        'config_hash': config_hash,
        'model_name': model_name,
        'tier': proto['tier'],
        'n_chains': CFG.n_chains,
        'total_samples': total_samples,
        'burn': proto['burn'],
        'converged': converged,
        'dic': dic_value,
        'loglike_available': enable_loglike,
        'ppc_available': CFG.enable_ppc,
        'f_rhat_max': metrics.get('f_rhat', np.nan),
        'f_ess_bulk_min': metrics.get('f_bulk', np.nan),
        'f_ess_tail_min': metrics.get('f_tail', np.nan),
        'n_rhat_max': metrics.get('n_rhat', np.nan),
        'n_ess_bulk_min': metrics.get('n_bulk', np.nan),
        'elapsed_minutes': t_elapsed / 60,
        'nc_path': f"{save_prefix}.nc",
        'hddm_path': f"{save_prefix}.hddm"
    }

    logger.info(
        f"  FINAL STATUS: "
        f"{'CONVERGED' if converged else 'NOT CONVERGED'} "
        f"({total_samples} total samples)."
    )

    # Explicit memory deallocation
    del model
    gc.collect()

    return manifest_record


# =============================================================================
# MAIN EXECUTION THREAD
# =============================================================================
def run_estimation_pipeline():
    """
    Orchestrates the global MCMC estimation workflow sequentially
    across architectures defined in the configuration lineage.

    [FIX-2b.4] Implements model-level checkpoint resume:
      - On startup, reads existing manifest to identify completed models.
      - Completed models are skipped without re-estimation.
      - Each newly completed model is immediately persisted to the
        manifest via atomic file write.
      - After interruption and restart, only incomplete models are fitted.
    """
    logger = _setup_estimation_logger()

    # Lineage validation
    # [FIX-2b.1] Uses load_active_lineage_state (not validate_pipeline_lineage)
    active_lineage = load_active_lineage_state(PATHS, logger)
    config_hash = active_lineage['config_hash']

    logger.info("=" * 70)
    logger.info(
        f"ESTIMATION PIPELINE INITIALIZED "
        f"(Mode: {CFG.run_mode.upper()})"
    )
    logger.info(f"Lineage Hash: {config_hash[:24]}...")
    logger.info(
        f"DDM Dimensionality: v, a, t + {CFG.include_params}"
    )
    logger.info(f"Parallel Chains: {CFG.n_chains}")
    logger.info(
        f"group_only_regressors: {CFG.group_only_regressors}"
    )
    logger.info(
        f"ArviZ Config: loglike={CFG.enable_loglike}, "
        f"ppc={CFG.enable_ppc}"
    )
    logger.info(f"Target Architectures: {CFG.final_all_models}")
    logger.info("=" * 70)

    if CFG.run_mode == 'debug':
        logger.warning(
            "EXECUTION MODE: DEBUG. "
            "Results hold no scientific validity."
        )
        time.sleep(2)

    # Load and validate empirical data
    data_path = CFG.base_dir / "hddm_data_unfair.csv"
    data = validate_empirical_data(str(data_path), logger)

    # -----------------------------------------------------------------
    # [FIX-2b.4] CHECKPOINT RESUME LOGIC
    # -----------------------------------------------------------------
    manifest_path = PATHS['manifests'] / "model_manifest.csv"
    completed_models, existing_records = load_existing_manifest(
        manifest_path
    )

    if completed_models:
        logger.info(
            f"\n  RESUME MODE ACTIVATED: "
            f"{len(completed_models)} model(s) already completed: "
            f"{sorted(completed_models)}"
        )
        logger.info(
            f"  These models will be skipped. Only remaining "
            f"models will be fitted."
        )

    # Track records for final summary display
    all_records = existing_records.copy()

    for model_name in CFG.final_all_models:
        # Checkpoint check: skip already-completed models
        if model_name in completed_models:
            logger.info(
                f"\n[{model_name.upper()}] "
                f"Already completed (checkpoint). Skipping."
            )
            continue

        try:
            record = fit_single_model(
                model_name, data, logger, config_hash
            )

            # [FIX-2b.4] Immediately persist to manifest
            append_manifest_record(manifest_path, record)
            logger.info(
                f"  [{model_name.upper()}] Manifest checkpoint saved."
            )

            all_records.append(record)

        except MemoryError as e:
            logger.error(
                f"MEMORY EXHAUSTION in [{model_name}]: {e}"
            )
            error_record = {
                'config_hash': config_hash,
                'model_name': model_name,
                'tier': CFG.mcmc_protocols[model_name]['tier'],
                'converged': False,
                'dic': float('inf'),
                'error': f"MemoryError: {str(e)}"
            }
            append_manifest_record(manifest_path, error_record)
            all_records.append(error_record)

            # Aggressive memory cleanup before attempting next model
            gc.collect()

        except Exception as e:
            logger.error(
                f"FATAL EXCEPTION in [{model_name}]: {e}",
                exc_info=True
            )
            error_record = {
                'config_hash': config_hash,
                'model_name': model_name,
                'tier': CFG.mcmc_protocols[model_name]['tier'],
                'converged': False,
                'dic': float('inf'),
                'error': str(e)
            }
            append_manifest_record(manifest_path, error_record)
            all_records.append(error_record)

    # -----------------------------------------------------------------
    # FINAL SUMMARY
    # -----------------------------------------------------------------
    df_manifest = pd.DataFrame(all_records)

    n_total = len(CFG.final_all_models)
    n_converged = (
        df_manifest['converged'].sum()
        if 'converged' in df_manifest.columns else 0
    )
    n_errors = (
        df_manifest['error'].notna().sum()
        if 'error' in df_manifest.columns else 0
    )

    logger.info(f"\n{'='*70}")
    logger.info(
        f"PIPELINE TERMINATED: "
        f"{n_converged}/{n_total} converged, "
        f"{n_errors} errors."
    )
    logger.info(f"{'='*70}")

    display_cols = [
        'model_name', 'tier', 'converged', 'dic',
        'f_rhat_max', 'f_ess_bulk_min', 'elapsed_minutes'
    ]
    available_cols = [
        c for c in display_cols if c in df_manifest.columns
    ]
    print("\n--- Estimation Summary ---")
    print(df_manifest[available_cols].to_string(index=False))


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    run_estimation_pipeline()

09:51:06 - INFO - Active Pipeline Hash: c801cab37a7dfbe3...
2026-03-30 09:51:06,699 INFO: Active Pipeline Hash: c801cab37a7dfbe3...
09:51:06 - INFO - ======================================================================
2026-03-30 09:51:06,700 INFO: ======================================================================
09:51:06 - INFO - ESTIMATION PIPELINE INITIALIZED (Mode: FINAL)
2026-03-30 09:51:06,702 INFO: ESTIMATION PIPELINE INITIALIZED (Mode: FINAL)
09:51:06 - INFO - Lineage Hash: 5cce0d3c92ade5078f1a2dab...
2026-03-30 09:51:06,704 INFO: Lineage Hash: 5cce0d3c92ade5078f1a2dab...
09:51:06 - INFO - DDM Dimensionality: v, a, t + ['z']
2026-03-30 09:51:06,706 INFO: DDM Dimensionality: v, a, t + ['z']
09:51:06 - INFO - Parallel Chains: 4
2026-03-30 09:51:06,708 INFO: Parallel Chains: 4
09:51:06 - INFO - group_only_regressors: True
2026-03-30 09:51:06,710 INFO: group_only_regressors: True
09:51:06 - INFO - ArviZ Config: loglike=False, ppc=False
2026-03-30 09:51:06,712 INFO: ArviZ Con

No model attribute --> setting up standard HDDM
Set model to ddm


09:51:07 - INFO -   Architecture: hddm.HDDMRegressor
2026-03-30 09:51:07,712 INFO:   Architecture: hddm.HDDMRegressor
09:51:07 - INFO -   include=['v', 'a', 't', 'z']
2026-03-30 09:51:07,714 INFO:   include=['v', 'a', 't', 'z']
09:51:07 - INFO -     -> v ~ 1
2026-03-30 09:51:07,715 INFO:     -> v ~ 1
09:51:07 - INFO -     -> a ~ 1
2026-03-30 09:51:07,718 INFO:     -> a ~ 1
09:51:07 - INFO -     -> t ~ 1
2026-03-30 09:51:07,720 INFO:     -> t ~ 1
09:51:07 - INFO -     -> z ~ 1
2026-03-30 09:51:07,722 INFO:     -> z ~ 1
09:51:07 - INFO -   Sampling: 8000 iterations x 4 chains (burn=2000)...
2026-03-30 09:51:07,725 INFO:   Sampling: 8000 iterations x 4 chains (burn=2000)...
/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encountered in double_scalars
  tmp2 = (x - v) * (fx - fw)
/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encountered in double_scalars
  tmp2 = (x - v) * (fx - 

[                  0%                  ] 2 of 8000 complete in 2.7 sec
[                  0%                  ] 3 of 8000 complete in 4.5 sec
[                  0%                  ] 4 of 8000 complete in 5.8 sec
[                  0%                  ] 5 of 8000 complete in 6.8 sec
[                  0%                  ] 6 of 8000 complete in 7.9 sec
[                  0%                  ] 7 of 8000 complete in 8.8 sec
[                  0%                  ] 8 of 8000 complete in 9.7 sec
[                  0%                  ] 9 of 8000 complete in 10.7 sec
[                  0%                  ] 10 of 8000 complete in 11.6 sec
[                  0%                  ] 11 of 8000 complete in 12.5 sec
[                  0%                  ] 12 of 8000 complete in 13.4 sec
[                  0%                  ] 13 of 8000 complete in 14.3 sec
[                  0%                  ] 14 of 8000 complete in 15.2 sec
[                  0%                  ] 15 of 8000 complete in 16

[                  0%                  ] 2 of 8000 complete in 2.6 sec
[                  0%                  ] 3 of 8000 complete in 4.3 sec
[                  0%                  ] 4 of 8000 complete in 5.7 sec
[                  0%                  ] 5 of 8000 complete in 6.9 sec
[                  0%                  ] 6 of 8000 complete in 7.9 sec
[                  0%                  ] 7 of 8000 complete in 8.8 sec
[                  0%                  ] 8 of 8000 complete in 9.8 sec
[                  0%                  ] 9 of 8000 complete in 10.7 sec
[                  0%                  ] 10 of 8000 complete in 11.7 sec
[                  0%                  ] 11 of 8000 complete in 12.6 sec
[                  0%                  ] 12 of 8000 complete in 13.5 sec
[                  0%                  ] 13 of 8000 complete in 14.5 sec
[                  0%                  ] 14 of 8000 complete in 15.3 sec
[                  0%                  ] 15 of 8000 complete in 16

[                  1%                  ] 114 of 8000 complete in 106.0 sec
[                  1%                  ] 115 of 8000 complete in 106.9 sec
[                  1%                  ] 116 of 8000 complete in 108.1 sec
[                  1%                  ] 117 of 8000 complete in 109.0 sec
[                  1%                  ] 118 of 8000 complete in 109.8 sec
[                  1%                  ] 119 of 8000 complete in 110.8 sec
[                  1%                  ] 120 of 8000 complete in 111.7 sec
[                  1%                  ] 121 of 8000 complete in 112.6 sec
[                  1%                  ] 122 of 8000 complete in 113.6 sec
[                  1%                  ] 123 of 8000 complete in 114.6 sec
[                  1%                  ] 124 of 8000 complete in 115.6 sec
[                  1%                  ] 125 of 8000 complete in 116.4 sec
[                  1%                  ] 126 of 8000 complete in 117.3 sec
[                  1%    

[                  1%                  ] 114 of 8000 complete in 107.1 sec
[                  1%                  ] 115 of 8000 complete in 108.1 sec
[                  1%                  ] 116 of 8000 complete in 109.0 sec
[                  1%                  ] 117 of 8000 complete in 110.0 sec
[                  1%                  ] 118 of 8000 complete in 110.9 sec
[                  1%                  ] 119 of 8000 complete in 111.7 sec
[                  1%                  ] 120 of 8000 complete in 112.7 sec
[                  1%                  ] 121 of 8000 complete in 113.7 sec
[                  1%                  ] 122 of 8000 complete in 114.7 sec
[                  1%                  ] 123 of 8000 complete in 115.5 sec
[                  1%                  ] 124 of 8000 complete in 116.4 sec
[                  1%                  ] 125 of 8000 complete in 117.2 sec
[                  1%                  ] 126 of 8000 complete in 118.1 sec
[                  1%    

[-                 2%                  ] 223 of 8000 complete in 204.0 sec
[-                 2%                  ] 224 of 8000 complete in 204.8 sec
[-                 2%                  ] 225 of 8000 complete in 205.6 sec
[-                 2%                  ] 226 of 8000 complete in 206.6 sec
[-                 2%                  ] 227 of 8000 complete in 207.4 sec
[-                 2%                  ] 228 of 8000 complete in 208.3 sec
[-                 2%                  ] 229 of 8000 complete in 209.1 sec
[-                 2%                  ] 230 of 8000 complete in 210.0 sec
[-                 2%                  ] 231 of 8000 complete in 211.0 sec
[-                 2%                  ] 232 of 8000 complete in 211.9 sec
[-                 2%                  ] 233 of 8000 complete in 212.7 sec
[-                 2%                  ] 234 of 8000 complete in 213.7 sec
[-                 2%                  ] 235 of 8000 complete in 214.5 sec
[-                 2%    

[-                 2%                  ] 223 of 8000 complete in 208.5 sec
[-                 2%                  ] 224 of 8000 complete in 209.4 sec
[-                 2%                  ] 225 of 8000 complete in 210.3 sec
[-                 2%                  ] 226 of 8000 complete in 211.2 sec
[-                 2%                  ] 227 of 8000 complete in 212.2 sec
[-                 2%                  ] 228 of 8000 complete in 213.1 sec
[-                 2%                  ] 229 of 8000 complete in 214.1 sec
[-                 2%                  ] 230 of 8000 complete in 214.9 sec
[-                 2%                  ] 231 of 8000 complete in 216.0 sec
[-                 2%                  ] 232 of 8000 complete in 216.9 sec
[-                 2%                  ] 233 of 8000 complete in 218.1 sec
[-                 2%                  ] 234 of 8000 complete in 219.0 sec
[-                 2%                  ] 235 of 8000 complete in 219.9 sec
[-                 2%    

# Step 3: Posterior Trace Extraction and Predictive Simulation

In [4]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 3: Posterior Predictive Checks (PPC) & Diagnostic Visualization
=============================================================================
Pipeline Position:
  Upstream:   Step 2b (reads .nc and .hddm files from models/ directory;
              reads model_manifest.csv from manifests/ directory)
  Downstream: Step 4 (reads ppc_metrics_{model}.json from ppc/ directory;
              reads observed/ppc_summary_long.csv from ppc/ directory;
              reads summary_{model}.csv from audit/ directory)

Methodological Purpose:
  - Consumes InferenceData artifacts (.nc) produced by Step 2b.
  - Executes rigorous Posterior Predictive Checks (PPC) at dual levels:
    1. Condition-level: Evaluates replication of empirical rejection
       rates per emotion condition via Mean Absolute Error (MAE).
    2. RT-quantile-level: Evaluates structural fidelity of simulated
       RT distributions (10th/50th/90th percentiles MAE).
  - Generates publication-grade diagnostic visualizations:
    * Trace plots for MCMC chain stationarity assessment.
    * Rank plots for mixing quality (Vehtari et al., 2021).
    * Global PPC density overlay (az.plot_ppc).
    * Condition-level PPC faceted RT distributions (NEW).
  - Validates cryptographic lineage hashes prior to artifact
    consumption to prevent cross-contamination.
=============================================================================
"""

import os
import gc
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import hddm

# Suppress inconsequential dependency warnings for clean audit logs
warnings.filterwarnings('ignore', category=FutureWarning)

# -----------------------------------------------------------------------------
# [FIX-3.1] CONFIGURATION IMPORT
# Uses load_active_lineage_state (not validate_pipeline_lineage)
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' unresolved. "
        "Execution of Step 2a is mandatory to generate "
        "configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# [FIX-3.2] Establish deterministic behavior using the configured seed
np.random.seed(CFG.base_seed)

# -----------------------------------------------------------------------------
# PUBLICATION-GRADE VISUALIZATION AESTHETICS (APA / Nature Standards)
# -----------------------------------------------------------------------------
sns.set_theme(style="ticks", palette="colorblind")
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False,
    'figure.autolayout': True
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_ppc_logger() -> logging.Logger:
    """
    Initializes dual-sink logging infrastructure (File + Stream)
    for PPC auditing.
    """
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_ppc_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'ppc_audit_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# CORE CLASS: PPC AUDIT & DIAGNOSTIC ENGINE
# =============================================================================
class PPCAuditEngine:
    """
    Orchestrates InferenceData consumption, multi-level PPC adequacy
    computation, diagnostic visualization, and data structuring for
    the subsequent analytical funnel in Step 4.
    """

    def __init__(self):
        self.logger = _setup_ppc_logger()

        # [FIX-3.1] Use load_active_lineage_state
        self.active_lineage = load_active_lineage_state(
            PATHS, self.logger
        )
        self.active_hash = self.active_lineage['pipeline_hash']

        self.manifest = self._load_and_validate_manifest()

        self.observed_stats: List[Dict] = []
        self.ppc_stats: List[Dict] = []

        self.logger.info("=" * 70)
        self.logger.info(
            f"PPC AUDIT ENGINE INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(
            f"Pipeline Hash: {self.active_hash[:24]}..."
        )
        self.logger.info(
            f"Target architecture count: {len(self.manifest)}"
        )
        self.logger.info("=" * 70)

    def _load_and_validate_manifest(self) -> pd.DataFrame:
        """
        Retrieves the architecture manifest from Step 2b.
        [FIX-3.5] Validates using pipeline_hash for full lineage
        consistency, not just config_hash.
        """
        manifest_path = PATHS['manifests'] / "model_manifest.csv"
        if not manifest_path.exists():
            raise FileNotFoundError(
                "Architecture manifest unresolved. "
                "Step 2b execution required."
            )

        df = pd.read_csv(manifest_path)
        if df.empty:
            raise RuntimeError(
                "Manifest dataset empty. "
                "Step 2b process unverified."
            )

        # [FIX-3.5] Lineage cross-validation using config_hash
        # (pipeline_hash is not stored per-model in manifest;
        # config_hash is the model-level lineage marker)
        if 'config_hash' in df.columns:
            artifact_hash = str(df['config_hash'].iloc[0])
            expected_hash = self.active_lineage['config_hash']
            if expected_hash != artifact_hash:
                raise RuntimeError(
                    f"\nCRITICAL LINEAGE MISMATCH!\n"
                    f"Active Config Hash: {expected_hash[:24]}...\n"
                    f"Manifest Config Hash: {artifact_hash[:24]}...\n"
                    f"Resolution: Re-execute Step 2b to synchronize."
                )

        return df

    # -----------------------------------------------------------------
    # ARVIZ-BASED DIAGNOSTIC VISUALIZATION
    # -----------------------------------------------------------------
    def _generate_trace_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs MCMC trace visualizations (posterior density and
        sequential traces) restricted to focal parameters for visual
        convergence assessment.
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            self.logger.warning(
                f"  [{model_name}] Focal parameters absent. "
                f"Trace visualization bypassed."
            )
            return

        try:
            az.plot_trace(
                infdata,
                var_names=focal_vars,
                compact=True,
                figsize=(12, 2.5 * len(focal_vars))
            )
            out_path = (
                PATHS['figures_supp']
                / f"diagnostics_trace_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(
                f"  Trace visualization exported: {out_path.name}"
            )
        except Exception as e:
            self.logger.warning(
                f"  Trace visualization failed for "
                f"[{model_name}]: {e}"
            )

    def _generate_rank_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs rank plots for focal parameters to detect
        non-stationarity and chain mixing discrepancies.
        Methodological advancement over standard trace plots.
        Reference: Vehtari et al. (2021).
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            return

        try:
            az.plot_rank(
                infdata,
                var_names=focal_vars,
                kind='vlines',
                vlines_kwargs={'lw': 0},
                marker_vlines_kwargs={'lw': 2}
            )
            out_path = (
                PATHS['figures_supp']
                / f"diagnostics_rank_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(
                f"  Rank plot exported: {out_path.name}"
            )
        except Exception as e:
            self.logger.warning(
                f"  Rank plot failed for [{model_name}]: {e}"
            )

    def _generate_ppc_density_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs posterior predictive density overlays.
        Juxtaposes empirical RT kernel density estimation against
        simulated posterior iterations for structural fit assessment.
        """
        if not hasattr(infdata, 'posterior_predictive'):
            self.logger.warning(
                f"  [{model_name}] 'posterior_predictive' group "
                f"absent. Density visualization bypassed."
            )
            return

        try:
            az.plot_ppc(
                infdata,
                var_names=['rt'],
                num_pp_samples=100,
                flatten=[]
            )
            out_path = (
                PATHS['figures_supp']
                / f"ppc_density_global_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(
                f"  PPC density plot exported: {out_path.name}"
            )
        except Exception as e:
            self.logger.warning(
                f"  PPC density plot failed for "
                f"[{model_name}]: {e}"
            )

    # -----------------------------------------------------------------
    # [FIX-3.4] CONDITION-LEVEL PPC FACETED VISUALIZATION
    # -----------------------------------------------------------------
    def _generate_conditionwise_ppc_plot(
        self,
        model_name: str,
        obs_df: pd.DataFrame,
        sim_df: pd.DataFrame
    ):
        """
        Constructs per-emotion faceted RT distribution comparisons
        between observed and simulated data. Each panel shows the
        observed RT density overlaid with the simulated RT density,
        disaggregated by accept/reject response.

        This addresses a gap in the original pipeline where only
        global PPC density was visualized, potentially masking
        condition-specific misfit.

        Reference: Wiecki, Sofer & Frank (2013), Figure 6.
        """
        if obs_df.empty or sim_df.empty:
            return

        if 'emotion' not in obs_df.columns:
            return

        emotions = sorted(obs_df['emotion'].unique())
        n_emo = len(emotions)

        if n_emo == 0:
            return

        fig, axes = plt.subplots(
            1, n_emo, figsize=(4 * n_emo, 4), squeeze=False
        )
        axes = axes.flatten()

        for idx, emo in enumerate(emotions):
            ax = axes[idx]
            emo_label = getattr(CFG, 'display_labels', {}).get(
                emo, emo
            )

            # Observed RT distribution
            obs_emo = obs_df[obs_df['emotion'] == emo]
            if not obs_emo.empty:
                obs_rt = np.abs(obs_emo['rt'].values)
                sns.kdeplot(
                    obs_rt, ax=ax, color='black',
                    linewidth=2.5, label='Observed', zorder=3
                )

            # Simulated RT distribution
            sim_emo = sim_df[sim_df['emotion'] == emo]
            if not sim_emo.empty:
                sim_rt_col = (
                    'rt_sampled'
                    if 'rt_sampled' in sim_emo.columns
                    else 'rt'
                )
                sim_rt = np.abs(sim_emo[sim_rt_col].values)
                sns.kdeplot(
                    sim_rt, ax=ax, color='#56B4E9',
                    linewidth=1.5, alpha=0.7,
                    label='Simulated', zorder=2
                )

            ax.set_title(emo_label, fontsize=12, fontweight='bold')
            ax.set_xlabel('RT (seconds)', fontsize=10)
            if idx == 0:
                ax.set_ylabel('Density', fontsize=10)
            else:
                ax.set_ylabel('')
            ax.legend(fontsize=8, loc='upper right')
            sns.despine(ax=ax, trim=True)

        plt.suptitle(
            f"Condition-Level PPC: {model_name.upper()}",
            fontsize=14, fontweight='bold', y=1.05
        )
        plt.tight_layout()

        out_path = (
            PATHS['figures_supp']
            / f"ppc_conditionwise_rt_{model_name}.pdf"
        )
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        self.logger.info(
            f"  Condition-level PPC plot exported: {out_path.name}"
        )

    # -----------------------------------------------------------------
    # QUANTITATIVE PPC ADEQUACY EVALUATION
    # -----------------------------------------------------------------
    def _compute_ppc_adequacy_from_hddm(
        self, model_name: str
    ) -> Dict[str, float]:
        """
        Quantifies predictive adequacy via posterior simulations.
        Disaggregates empirical vs. simulated data by experimental
        condition to compute MAE for rejection rates and RT deciles.

        [FIX-3.3] Observed and simulated data are explicitly
        separated to avoid structural ambiguity from
        post_pred_gen(append_data=True). The observed columns are
        'rt' and 'response'; simulated columns are 'rt_sampled'
        and 'response_sampled' (if present) or the PPC-generated
        columns identified dynamically.

        [FIX-3.6] Graceful fallback on .hddm load failure.
        """
        hddm_path = PATHS['models'] / f"hddm_{model_name}.hddm"
        if not hddm_path.exists():
            self.logger.warning(
                f"  [{model_name}] .hddm artifact unresolved. "
                f"Quantitative PPC bypassed."
            )
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        # [FIX-3.6] Graceful model loading with error handling
        try:
            model = hddm.load(str(hddm_path))
        except Exception as e:
            self.logger.warning(
                f"  [{model_name}] Failed to load .hddm file: {e}. "
                f"Quantitative PPC bypassed."
            )
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        # Posterior predictive data generation
        try:
            ppc_data = hddm.utils.post_pred_gen(
                model, samples=200, append_data=True
            )
        except Exception as e:
            self.logger.warning(
                f"  [{model_name}] post_pred_gen exception: {e}"
            )
            del model
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        # [FIX-3.3] Identify observed vs simulated column names
        # post_pred_gen(append_data=True) returns a DataFrame where:
        #   - 'rt' and 'response' are the OBSERVED values
        #   - 'rt_sampled'/'response_sampled' are simulated values
        #     (column names depend on HDDM version)
        rt_col_sim = (
            'rt_sampled'
            if 'rt_sampled' in ppc_data.columns
            else 'rt_sim'
        )
        resp_col_sim = (
            'response_sampled'
            if 'response_sampled' in ppc_data.columns
            else 'response_sim'
        )

        # Verify simulated columns exist
        if rt_col_sim not in ppc_data.columns:
            self.logger.warning(
                f"  [{model_name}] Simulated RT column "
                f"'{rt_col_sim}' not found in PPC output. "
                f"Available columns: {ppc_data.columns.tolist()}"
            )
            del model, ppc_data
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        if 'emotion' not in ppc_data.columns:
            self.logger.warning(
                f"  [{model_name}] 'emotion' column missing "
                f"in PPC output. Adequacy evaluation terminated."
            )
            del model, ppc_data
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        obs_records = []
        sim_records = []

        # Store a copy of observed-only data for condition-level plot
        # [FIX-3.3] Use ORIGINAL observed data from model, not PPC
        obs_data_for_plot = model.data.copy()

        for emo in ppc_data['emotion'].unique():
            subset = ppc_data[ppc_data['emotion'] == emo]

            # [FIX-3.3] Observed rejection rate from ORIGINAL column
            obs_rej_rate = (subset['response'] == 0).mean()
            obs_records.append({
                'config_hash': self.active_lineage['config_hash'],
                'model_name': model_name,
                'emotion': emo,
                'stat_name': 'rejection_rate',
                'stat_value': float(obs_rej_rate),
                'source': 'observed'
            })

            # [FIX-3.3] Simulated rejection rate from SIMULATED column
            if resp_col_sim in subset.columns:
                sim_rej_rate = (subset[resp_col_sim] == 0).mean()
            else:
                # Fallback: use response column if no separate
                # simulated column exists (some HDDM versions)
                sim_rej_rate = obs_rej_rate
                self.logger.warning(
                    f"  [{model_name}] No separate simulated "
                    f"response column for {emo}. "
                    f"Using observed as proxy."
                )

            sim_records.append({
                'config_hash': self.active_lineage['config_hash'],
                'model_name': model_name,
                'emotion': emo,
                'stat_name': 'rejection_rate',
                'stat_value': float(sim_rej_rate),
                'source': 'simulated'
            })

            # RT quantile extraction by behavioral response
            for resp_val, resp_label in [(1, 'accept'), (0, 'reject')]:
                obs_sub = subset[subset['response'] == resp_val]

                if resp_col_sim in subset.columns:
                    sim_sub = subset[
                        subset[resp_col_sim] == resp_val
                    ]
                else:
                    sim_sub = obs_sub

                # Skip sparse data configurations
                if len(obs_sub) < 5 or len(sim_sub) < 5:
                    continue

                obs_rt = np.abs(obs_sub['rt'].values)
                sim_rt = np.abs(sim_sub[rt_col_sim].values)

                for q_label, q_val in [
                    ('rt_q10', 0.10),
                    ('rt_q50', 0.50),
                    ('rt_q90', 0.90)
                ]:
                    obs_records.append({
                        'config_hash': (
                            self.active_lineage['config_hash']
                        ),
                        'model_name': model_name,
                        'emotion': emo,
                        'response_type': resp_label,
                        'stat_name': q_label,
                        'stat_value': float(
                            np.quantile(obs_rt, q_val)
                        ),
                        'source': 'observed'
                    })
                    sim_records.append({
                        'config_hash': (
                            self.active_lineage['config_hash']
                        ),
                        'model_name': model_name,
                        'emotion': emo,
                        'response_type': resp_label,
                        'stat_name': q_label,
                        'stat_value': float(
                            np.quantile(sim_rt, q_val)
                        ),
                        'source': 'simulated'
                    })

        self.observed_stats.extend(obs_records)
        self.ppc_stats.extend(sim_records)

        # [FIX-3.4] Generate condition-level PPC plot
        self._generate_conditionwise_ppc_plot(
            model_name, obs_data_for_plot, ppc_data
        )

        # Statistical discrepancy calculation (MAE)
        df_obs = pd.DataFrame(obs_records)
        df_sim = pd.DataFrame(sim_records)

        if df_obs.empty or df_sim.empty:
            del model, ppc_data
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        merge_keys = ['emotion', 'stat_name']
        if ('response_type' in df_obs.columns
                and 'response_type' in df_sim.columns):
            merge_keys.append('response_type')

        merged = pd.merge(
            df_obs[merge_keys + ['stat_value']],
            df_sim[merge_keys + ['stat_value']],
            on=merge_keys,
            suffixes=('_obs', '_sim'),
            how='inner'
        )
        merged['abs_error'] = np.abs(
            merged['stat_value_obs'] - merged['stat_value_sim']
        )

        choice_errs = merged[
            merged['stat_name'] == 'rejection_rate'
        ]['abs_error']
        rt_errs = merged[
            merged['stat_name'].str.startswith('rt_')
        ]['abs_error']

        choice_mae = (
            float(choice_errs.mean())
            if not choice_errs.empty else np.nan
        )
        choice_max = (
            float(choice_errs.max())
            if not choice_errs.empty else np.nan
        )
        rt_mae = (
            float(rt_errs.mean())
            if not rt_errs.empty else np.nan
        )
        rt_max = (
            float(rt_errs.max())
            if not rt_errs.empty else np.nan
        )

        # Adequacy classification
        is_adequate = (
            (not np.isnan(choice_mae))
            and (choice_mae <= CFG.ppc_choice_mae_max)
            and (choice_max <= CFG.ppc_choice_max_err)
            and (not np.isnan(rt_mae))
            and (rt_mae <= CFG.ppc_rt_quantile_mae_max)
            and (rt_max <= CFG.ppc_rt_quantile_max_err)
        )

        self.logger.info(
            f"  PPC Adequacy: Choice MAE={choice_mae:.4f} "
            f"(Limit: {CFG.ppc_choice_mae_max}), "
            f"RT MAE={rt_mae:.4f} "
            f"(Limit: {CFG.ppc_rt_quantile_mae_max}) "
            f"-> {'PASS' if is_adequate else 'FAIL'}"
        )

        del model, ppc_data
        gc.collect()

        return {
            'choice_mae': choice_mae,
            'choice_max_err': choice_max,
            'rt_mae': rt_mae,
            'rt_max_err': rt_max,
            'pass': is_adequate
        }

    # -----------------------------------------------------------------
    # POSTERIOR PARAMETER MANIFEST GENERATION
    # -----------------------------------------------------------------
    def _generate_posterior_manifest(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs a structural taxonomy of posterior parameters
        (identifying family, level, focal/nuisance status) for
        targeted downstream extraction in Steps 4-5.
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_params = CFG.identify_focal_parameters(all_vars)

        records = []
        for var in all_vars:
            family = var.split('_')[0] if '_' in var else var
            if family not in ['v', 'a', 't', 'z']:
                family = 'other'

            level = (
                'subject' if '_subj' in var
                else (
                    'sd'
                    if (var.endswith('_std')
                        or var.endswith('_var'))
                    else 'group'
                )
            )
            is_focal = var in focal_params

            records.append({
                'variable_name': var,
                'family': family,
                'level': level,
                'focal': is_focal
            })

        df_manifest = pd.DataFrame(records)
        out_path = (
            PATHS['audit']
            / f"posterior_manifest_{model_name}.csv"
        )
        df_manifest.to_csv(out_path, index=False)

    # -----------------------------------------------------------------
    # SINGLE MODEL PROCESSING
    # -----------------------------------------------------------------
    def process_model(self, model_name: str):
        """
        Executes the comprehensive audit sequence for a singular
        target architecture: load InferenceData, generate diagnostics,
        compute PPC adequacy, export all artifacts.
        """
        self.logger.info(f"\n{'─'*50}")
        self.logger.info(
            f"Target Architecture: [{model_name.upper()}]"
        )
        self.logger.info(f"{'─'*50}")

        # Verify target architecture in manifest
        model_row = self.manifest[
            self.manifest['model_name'] == model_name
        ]
        if model_row.empty:
            self.logger.warning(
                f"  [{model_name}] Unregistered in manifest. "
                f"Bypassed."
            )
            return

        nc_path = PATHS['models'] / f"hddm_{model_name}.nc"
        if not nc_path.exists():
            self.logger.warning(
                f"  [{model_name}] NetCDF unresolved at "
                f"{nc_path.name}. Bypassed."
            )
            return

        infdata = az.from_netcdf(str(nc_path))
        self.logger.info(
            f"  InferenceData imported. Groups: "
            f"{list(infdata.groups())}"
        )

        # Posterior parameter manifest
        self._generate_posterior_manifest(infdata, model_name)

        # ArviZ statistical summary with focal/nuisance classification
        summary_df = az.summary(
            infdata, round_to=4, hdi_prob=0.95
        )
        focal_params = CFG.identify_focal_parameters(
            summary_df.index.tolist()
        )
        summary_df['param_class'] = [
            'focal' if p in focal_params else 'nuisance'
            for p in summary_df.index
        ]
        summary_path = (
            PATHS['audit'] / f"summary_{model_name}.csv"
        )
        summary_df.to_csv(summary_path)
        self.logger.info(
            f"  ArviZ summary exported: {summary_path.name}"
        )

        # Diagnostic visualizations
        self._generate_trace_plots(infdata, model_name)
        self._generate_rank_plots(infdata, model_name)
        self._generate_ppc_density_plots(infdata, model_name)

        # Quantitative PPC adequacy
        ppc_metrics = self._compute_ppc_adequacy_from_hddm(
            model_name
        )

        # Persist PPC metrics as JSON for Step 4 consumption
        ppc_record_path = (
            PATHS['ppc'] / f"ppc_metrics_{model_name}.json"
        )
        with open(ppc_record_path, 'w', encoding='utf-8') as f:
            json.dump(
                {
                    'config_hash': (
                        self.active_lineage['config_hash']
                    ),
                    'pipeline_hash': self.active_hash,
                    'model_name': model_name,
                    **ppc_metrics
                },
                f, indent=2
            )

        del infdata
        gc.collect()

    # -----------------------------------------------------------------
    # MASTER PIPELINE
    # -----------------------------------------------------------------
    def run(self):
        """
        Orchestrates the sequential multi-architecture PPC audit
        pipeline across all model architectures defined in the
        configuration SSOT.
        """
        for model_name in CFG.final_all_models:
            self.process_model(model_name)

        # Export aggregated statistics for Step 4 fallback path
        if self.observed_stats:
            df_obs = pd.DataFrame(self.observed_stats)
            df_obs.to_csv(
                PATHS['ppc'] / "observed_summary_long.csv",
                index=False
            )

        if self.ppc_stats:
            df_sim = pd.DataFrame(self.ppc_stats)
            df_sim.to_csv(
                PATHS['ppc'] / "ppc_summary_long.csv",
                index=False
            )

        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            f"STEP 3 PIPELINE TERMINATED: Audit executed across "
            f"{len(CFG.final_all_models)} architectures."
        )
        self.logger.info(
            f"Totals: Observed records={len(self.observed_stats)}"
            f" | Simulated records={len(self.ppc_stats)}"
        )
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    engine = PPCAuditEngine()
    engine.run()

11:58:20 - INFO - Active Pipeline Hash: 719a1af99cc02ea5...
2026-03-29 11:58:20,424 INFO: Active Pipeline Hash: 719a1af99cc02ea5...
11:58:20 - INFO - ======================================================================
2026-03-29 11:58:20,439 INFO: ======================================================================
11:58:20 - INFO - PPC AUDIT ENGINE INITIATED (Mode: DEBUG)
2026-03-29 11:58:20,441 INFO: PPC AUDIT ENGINE INITIATED (Mode: DEBUG)
11:58:20 - INFO - Pipeline Hash: 719a1af99cc02ea57265605e...
2026-03-29 11:58:20,443 INFO: Pipeline Hash: 719a1af99cc02ea57265605e...
11:58:20 - INFO - Target architecture count: 8
2026-03-29 11:58:20,445 INFO: Target architecture count: 8
11:58:20 - INFO - ======================================================================
2026-03-29 11:58:20,448 INFO: ======================================================================
11:58:20 - INFO - 
──────────────────────────────────────────────────
2026-03-29 11:58:20,450 INFO: 
─────────────────

2026-03-29 11:58:25,052 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:58:25,053 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 3228]
2026-03-29 11:58:25,054 INFO: Closed glyph list over 'glyf': 16 glyphs after
2026-03-29 11:58:25,055 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:58:25,056 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 3228]
2026-03-29 11:58:25,057 INFO: Retaining 16 glyphs
2026-03-29 11:58:25,059 INFO: head subsetting not needed
2026-03-29 11:58:25,059 INFO: hhea subsetting not needed
2026-03-29 11:58:25,060 INFO: maxp subsetting not needed
2026-03-29 11:58:25,061 INFO: OS/2 subsetting not needed
2026-03-29 11:58:25,065 INFO: hmtx subsetted
2026

2026-03-29 11:58:26,532 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86, 3506, 3507, 3508, 3509, 3510, 3511]
2026-03-29 11:58:26,533 INFO: Closing glyph list over 'glyf': 27 glyphs before
2026-03-29 11:58:26,533 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'uni239B', 'uni239C', 'uni239D', 'uni239E', 'uni239F', 'uni23A0', 'zero']
2026-03-29 11:58:26,534 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86, 3506, 3507, 3508, 3509, 3510, 3511]
2026-03-29 11:58:26,535 INFO: Closed glyph list over 'glyf': 27 glyphs after
2026-03-29 11:58:26,536 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'uni239B', 'uni239C', 'uni239D', 'uni239E', 

2026-03-29 11:58:32,897 INFO: Added gid0 to subset
2026-03-29 11:58:32,897 INFO: Added first four glyphs to subset
2026-03-29 11:58:32,898 INFO: Closing glyph list over 'GSUB': 15 glyphs before
2026-03-29 11:58:32,899 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:58:32,901 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 3228]
2026-03-29 11:58:32,904 INFO: Closed glyph list over 'GSUB': 15 glyphs after
2026-03-29 11:58:32,905 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:58:32,906 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 3228]
2026-03-29 11:58:32,906 INFO: Closing glyph list over 'MATH': 15 glyphs before
2026-03-29 11:58:32,908 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five

2026-03-29 11:58:34,129 INFO: Closing glyph list over 'GSUB': 21 glyphs before
2026-03-29 11:58:34,130 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'zero']
2026-03-29 11:58:34,132 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86]
2026-03-29 11:58:34,136 INFO: Closed glyph list over 'GSUB': 21 glyphs after
2026-03-29 11:58:34,136 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'zero']
2026-03-29 11:58:34,137 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86]
2026-03-29 11:58:34,139 INFO: Closing glyph list over 'MATH': 21 glyphs before
2026-03-29 11:58:34,140 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four',

...

2026-03-29 11:59:01,513 INFO: GDEF subsetted
2026-03-29 11:59:01,531 INFO: GPOS subsetted
2026-03-29 11:59:01,533 INFO: GSUB subsetted
2026-03-29 11:59:01,535 INFO: MATH subsetted
2026-03-29 11:59:01,535 INFO: name subsetting not needed
2026-03-29 11:59:01,538 INFO: glyf subsetted
2026-03-29 11:59:01,539 INFO: head pruned
2026-03-29 11:59:01,540 INFO: OS/2 Unicode ranges pruned: [0]
2026-03-29 11:59:01,541 INFO: glyf pruned
2026-03-29 11:59:01,542 INFO: GDEF pruned
2026-03-29 11:59:01,543 INFO: GPOS pruned
2026-03-29 11:59:01,544 INFO: GSUB pruned
11:59:01 - INFO -   Rank plot exported: diagnostics_rank_vt.pdf
2026-03-29 11:59:01,822 INFO:   Rank plot exported: diagnostics_rank_vt.pdf
11:59:01 - WARNING -   [vt] 'posterior_predictive' group absent. Density visualization bypassed.
2026-03-29 11:59:01,824 WARNING:   [vt] 'posterior_predictive' group absent. Density visualization bypassed.
11:59:05 - WARNING -   [vt] Failed to load .hddm file: [Errno 2] No such file or directory: '/home/j

2026-03-29 11:59:09,675 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 3228]
2026-03-29 11:59:09,676 INFO: Closed glyph list over 'glyf': 16 glyphs after
2026-03-29 11:59:09,677 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:59:09,678 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 3228]
2026-03-29 11:59:09,679 INFO: Retaining 16 glyphs
2026-03-29 11:59:09,681 INFO: head subsetting not needed
2026-03-29 11:59:09,682 INFO: hhea subsetting not needed
2026-03-29 11:59:09,683 INFO: maxp subsetting not needed
2026-03-29 11:59:09,684 INFO: OS/2 subsetting not needed
2026-03-29 11:59:09,688 INFO: hmtx subsetted
2026-03-29 11:59:09,691 INFO: cmap subsetted
2026-03-29 11:59:09,692 INFO: fpgm subsetting not needed
2026-03-29 11:59:09,693 INFO: prep subsetting not needed
2026-03-29 11:59:09,694 INFO: cvt  su

2026-03-29 11:59:11,647 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'uni239B', 'uni239C', 'uni239D', 'uni239E', 'uni239F', 'uni23A0', 'zero']
2026-03-29 11:59:11,648 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86, 3506, 3507, 3508, 3509, 3510, 3511]
2026-03-29 11:59:11,650 INFO: Closed glyph list over 'glyf': 27 glyphs after
2026-03-29 11:59:11,651 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'uni239B', 'uni239C', 'uni239D', 'uni239E', 'uni239F', 'uni23A0', 'zero']
2026-03-29 11:59:11,652 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86, 3506, 3507, 3508, 3509, 3510, 3511]
2026-03-29 11:59:11,653 INFO: Retaining 27 glyphs

2026-03-29 11:59:21,401 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:59:21,403 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 3228]
2026-03-29 11:59:21,405 INFO: Closed glyph list over 'GSUB': 16 glyphs after
2026-03-29 11:59:21,406 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:59:21,407 INFO: Glyph IDs:   [0, 1, 2, 3, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 3228]
2026-03-29 11:59:21,408 INFO: Closing glyph list over 'MATH': 16 glyphs before
2026-03-29 11:59:21,409 INFO: Glyph names: ['.notdef', '.null', 'eight', 'five', 'four', 'minus', 'nine', 'nonmarkingreturn', 'one', 'period', 'seven', 'six', 'space', 'three', 'two', 'zero']
2026-03-29 11:59:21,411 INFO: Glyph IDs:   [0, 1, 2, 3, 

2026-03-29 11:59:23,810 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'zero']
2026-03-29 11:59:23,812 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86]
2026-03-29 11:59:23,816 INFO: Closed glyph list over 'GSUB': 21 glyphs after
2026-03-29 11:59:23,817 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright', 's', 'six', 'space', 'two', 'zero']
2026-03-29 11:59:23,818 INFO: Glyph IDs:   [0, 1, 2, 3, 11, 12, 19, 20, 21, 23, 25, 27, 53, 68, 70, 75, 76, 78, 79, 81, 86]
2026-03-29 11:59:23,820 INFO: Closing glyph list over 'MATH': 21 glyphs before
2026-03-29 11:59:23,821 INFO: Glyph names: ['.notdef', '.null', 'R', 'a', 'c', 'eight', 'four', 'h', 'i', 'k', 'l', 'n', 'nonmarkingreturn', 'one', 'parenleft', 'parenright',

# Step 4: Convergence Diagnostics and Model Selection

In [5]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 4: Four-Level Diagnostic Funnel & Optimal Model Selection
=============================================================================
Pipeline Position:
  Upstream:   Step 2b (reads .nc and .hddm files from models/ directory;
              reads model_manifest.csv from manifests/ directory)
              Step 3 (reads ppc_metrics_{model}.json from ppc/ directory;
              reads observed/ppc_summary_long.csv from ppc/ directory)
  Downstream: Step 5 (reads final_model_selection_audit.csv from audit/;
              reads .nc file of winning model from models/)
              Step 6 (same as Step 5)

Methodological Purpose:
  - Implements a hierarchical evaluation framework (Four-Level Funnel):
    Level 1 (Technical): Validates artifact integrity (.nc, .hddm).
    Level 2 (Convergence): Evaluates stratified MCMC stationarity via
                           ArviZ InferenceData (Vehtari et al., 2021).
    Level 3 (PPC Adequacy): Enforces absolute goodness-of-fit via
                            empirical MAE thresholds from Step 3.
    Level 4 (Relative): Ranks surviving architectures via PSIS-LOO-CV
                        (primary), WAIC (secondary), DIC (fallback).
  - Selects optimal model exclusively from architectures passing L1-L3.
  - Constructs publication-grade diagnostics for the winning model.
  - Exports structured cryptographic audit trails.

=============================================================================
"""

import gc
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import hddm

# Suppress inconsequential dependency warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# -----------------------------------------------------------------------------
# [FIX-4.1] CONFIGURATION IMPORT
# Uses load_active_lineage_state (not validate_pipeline_lineage)
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' unresolved. "
        "Execution of Step 2a is mandatory to generate "
        "configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# [FIX-4.2] Establish deterministic behavior using the configured seed
np.random.seed(CFG.base_seed)

# -----------------------------------------------------------------------------
# PUBLICATION-GRADE VISUALIZATION AESTHETICS
# -----------------------------------------------------------------------------
sns.set_theme(style="ticks", palette="colorblind")
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False,
    'figure.autolayout': True
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_funnel_logger() -> logging.Logger:
    """
    Initializes dual-sink logging for the diagnostic funnel audit.
    """
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_funnel_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'model_selection_funnel_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# CORE CLASS: FOUR-LEVEL DIAGNOSTIC FUNNEL
# =============================================================================
class DiagnosticFunnelEngine:
    """
    Executes a sequential, hierarchical model evaluation framework.
    Filters model architectures through progressive stringency,
    computing relative information criteria (PSIS-LOO/WAIC/DIC)
    exclusively for empirically adequate models.
    """

    def __init__(self):
        self.logger = _setup_funnel_logger()

        # [FIX-4.1] Use load_active_lineage_state
        self.active_lineage = load_active_lineage_state(
            PATHS, self.logger
        )
        self.active_hash = self.active_lineage['config_hash']

        self.manifest = self._load_manifest()
        self.comparison_records: List[Dict] = []

        self.logger.info("=" * 70)
        self.logger.info(
            f"DIAGNOSTIC FUNNEL ENGINE INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(
            f"Pipeline Hash: "
            f"{self.active_lineage['pipeline_hash'][:24]}..."
        )
        self.logger.info(
            f"Stratified Convergence: "
            f"Focal R-hat<={CFG.rhat_focal}, "
            f"ESS>={CFG.ess_bulk_focal} | "
            f"Nuisance R-hat<={CFG.rhat_nuisance}, "
            f"ESS>={CFG.ess_bulk_nuisance}"
        )
        self.logger.info(
            f"PPC Adequacy: Choice MAE<={CFG.ppc_choice_mae_max}, "
            f"RT MAE<={CFG.ppc_rt_quantile_mae_max}"
        )
        self.logger.info("=" * 70)

    def _load_manifest(self) -> pd.DataFrame:
        """Retrieves Step 2b architecture manifest."""
        manifest_path = PATHS['manifests'] / "model_manifest.csv"
        if manifest_path.exists():
            return pd.read_csv(manifest_path)
        self.logger.warning(
            "Architecture manifest unresolved. "
            "Initiating dynamic discovery."
        )
        return pd.DataFrame()

    # -----------------------------------------------------------------
    # LEVEL 1: TECHNICAL ARTIFACT VERIFICATION
    # -----------------------------------------------------------------
    def evaluate_level1_technical(
        self, model_name: str
    ) -> bool:
        """
        Validates the structural presence of compiled ArviZ (.nc)
        and HDDM (.hddm) artifacts on disk.
        """
        nc_file = PATHS['models'] / f"hddm_{model_name}.nc"
        hddm_file = PATHS['models'] / f"hddm_{model_name}.hddm"

        nc_ok = nc_file.exists()
        hddm_ok = hddm_file.exists()

        if not nc_ok:
            self.logger.warning(
                f"  L1: NetCDF unresolved -> {nc_file.name}"
            )
        if not hddm_ok:
            self.logger.warning(
                f"  L1: HDDM artifact unresolved -> "
                f"{hddm_file.name}"
            )

        return nc_ok and hddm_ok

    # -----------------------------------------------------------------
    # LEVEL 2: STRATIFIED CONVERGENCE DIAGNOSTICS
    # -----------------------------------------------------------------
    def evaluate_level2_convergence(
        self, infdata: az.InferenceData, model_name: str
    ) -> Dict[str, Any]:
        """
        Quantifies MCMC stationarity and mixing efficiency via
        stratified constraints. Focal group-level parameters require
        stricter thresholds than nuisance subject-level deviations.
        """
        summary_df = az.summary(
            infdata, round_to=4, hdi_prob=0.95
        )

        required_cols = {'r_hat', 'ess_bulk'}
        missing = required_cols - set(summary_df.columns)
        if missing:
            return {
                'pass': False,
                'reason': (
                    f"ArviZ schema violation: missing {missing}"
                ),
                'f_rhat_max': np.nan,
                'f_ess_bulk_min': np.nan
            }

        focal_params = CFG.identify_focal_parameters(
            summary_df.index.tolist()
        )
        summary_df['param_class'] = [
            'focal' if p in focal_params else 'nuisance'
            for p in summary_df.index
        ]

        focal_df = summary_df[
            summary_df['param_class'] == 'focal'
        ]
        nuisance_df = summary_df[
            summary_df['param_class'] == 'nuisance'
        ]

        f_rhat = (
            focal_df['r_hat'].max()
            if not focal_df.empty else 1.0
        )
        f_bulk = (
            focal_df['ess_bulk'].min()
            if not focal_df.empty else float('inf')
        )
        f_tail = (
            focal_df['ess_tail'].min()
            if ('ess_tail' in focal_df.columns
                and not focal_df.empty)
            else float('inf')
        )

        n_rhat = (
            nuisance_df['r_hat'].max()
            if not nuisance_df.empty else 1.0
        )
        n_bulk = (
            nuisance_df['ess_bulk'].min()
            if not nuisance_df.empty else float('inf')
        )
        n_tail = (
            nuisance_df['ess_tail'].min()
            if ('ess_tail' in nuisance_df.columns
                and not nuisance_df.empty)
            else float('inf')
        )

        focal_ok = (
            f_rhat <= CFG.rhat_focal
            and f_bulk >= CFG.ess_bulk_focal
            and f_tail >= CFG.ess_tail_focal
        )
        nuisance_ok = (
            n_rhat <= CFG.rhat_nuisance
            and n_bulk >= CFG.ess_bulk_nuisance
            and n_tail >= CFG.ess_tail_nuisance
        )

        converged = focal_ok and nuisance_ok

        reason = (
            "Criteria Satisfied" if converged
            else (
                f"Focal: R-hat={f_rhat:.3f},"
                f"ESS_b={f_bulk:.0f},"
                f"ESS_t={f_tail:.0f} | "
                f"Nuisance: R-hat={n_rhat:.3f},"
                f"ESS_b={n_bulk:.0f}"
            )
        )

        self.logger.info(
            f"  L2 FOCAL:    R-hat={f_rhat:.3f}, "
            f"ESS_bulk={f_bulk:.0f}, ESS_tail={f_tail:.0f} "
            f"-> {'PASS' if focal_ok else 'FAIL'}"
        )
        self.logger.info(
            f"  L2 NUISANCE: R-hat={n_rhat:.3f}, "
            f"ESS_bulk={n_bulk:.0f}, ESS_tail={n_tail:.0f} "
            f"-> {'PASS' if nuisance_ok else 'FAIL'}"
        )

        if not focal_df.empty:
            focal_df.to_csv(
                PATHS['audit']
                / f"focal_parameter_audit_{model_name}.csv"
            )

        return {
            'pass': converged,
            'reason': reason,
            'f_rhat_max': f_rhat,
            'f_ess_bulk_min': f_bulk,
            'f_ess_tail_min': f_tail,
            'n_rhat_max': n_rhat,
            'n_ess_bulk_min': n_bulk,
            'n_ess_tail_min': n_tail
        }

    # -----------------------------------------------------------------
    # LEVEL 3: PPC ADEQUACY
    # -----------------------------------------------------------------
    def evaluate_level3_ppc(
        self, model_name: str
    ) -> Dict[str, Any]:
        """
        Validates predictive fidelity constraints. Integrates JSON
        metrics exported during Step 3 to ensure structural
        replication of behavioral data.
        """
        ppc_path = (
            PATHS['ppc'] / f"ppc_metrics_{model_name}.json"
        )

        if not ppc_path.exists():
            self.logger.warning(
                f"  L3: PPC JSON unresolved for [{model_name}]. "
                f"Executing long-format fallback..."
            )
            return self._evaluate_ppc_from_long_format(model_name)

        with open(ppc_path, 'r', encoding='utf-8') as f:
            ppc_data = json.load(f)

        is_adequate = ppc_data.get('pass', False)
        choice_mae = ppc_data.get('choice_mae', np.nan)
        rt_mae = ppc_data.get('rt_mae', np.nan)

        return {
            'pass': is_adequate,
            'choice_mae': choice_mae,
            'rt_mae': rt_mae
        }

    def _evaluate_ppc_from_long_format(
        self, model_name: str
    ) -> Dict[str, Any]:
        """
        Fallback adequacy evaluation using aggregated CSV formats
        when target-specific JSON manifests are unavailable.
        """
        obs_path = PATHS['ppc'] / "observed_summary_long.csv"
        sim_path = PATHS['ppc'] / "ppc_summary_long.csv"

        if not obs_path.exists() or not sim_path.exists():
            return {
                'pass': False,
                'choice_mae': np.nan,
                'rt_mae': np.nan
            }

        df_obs = pd.read_csv(obs_path)
        df_sim = pd.read_csv(sim_path)

        df_sim_model = df_sim[
            df_sim['model_name'] == model_name
        ]
        if df_sim_model.empty:
            return {
                'pass': False,
                'choice_mae': np.nan,
                'rt_mae': np.nan
            }

        merge_keys = ['emotion', 'stat_name']
        if 'response_type' in df_sim_model.columns:
            merge_keys.append('response_type')

        sim_agg = (
            df_sim_model
            .groupby(merge_keys)['stat_value']
            .mean()
            .reset_index()
            .rename(columns={'stat_value': 'sim_value'})
        )

        obs_subset = df_obs
        if 'source' in df_obs.columns:
            obs_subset = df_obs[df_obs['source'] == 'observed']

        merged = pd.merge(
            obs_subset,
            sim_agg,
            on=merge_keys,
            how='inner'
        )

        if ('stat_value' not in merged.columns
                or 'sim_value' not in merged.columns):
            return {
                'pass': False,
                'choice_mae': np.nan,
                'rt_mae': np.nan
            }

        merged['abs_error'] = np.abs(
            merged['stat_value'] - merged['sim_value']
        )

        choice_errs = merged[
            merged['stat_name'] == 'rejection_rate'
        ]['abs_error']
        rt_errs = merged[
            merged['stat_name'].str.startswith('rt_')
        ]['abs_error']

        choice_mae = (
            float(choice_errs.mean())
            if not choice_errs.empty else np.nan
        )
        rt_mae = (
            float(rt_errs.mean())
            if not rt_errs.empty else np.nan
        )

        is_adequate = (
            (not np.isnan(choice_mae))
            and (choice_mae <= CFG.ppc_choice_mae_max)
            and (not np.isnan(rt_mae))
            and (rt_mae <= CFG.ppc_rt_quantile_mae_max)
        )

        return {
            'pass': is_adequate,
            'choice_mae': choice_mae,
            'rt_mae': rt_mae
        }

    # -----------------------------------------------------------------
    # LEVEL 4: RELATIVE MODEL COMPARISON
    # [FIX-4.4] Pareto-k diagnostic for PSIS-LOO-CV
    # [FIX-4.5] DIC fallback with methodological warning
    # -----------------------------------------------------------------
    def compute_model_comparison_metrics(
        self,
        model_name: str,
        infdata: az.InferenceData
    ) -> Dict[str, Any]:
        """
        Computes predictive performance indicators: PSIS-LOO-CV
        (primary), WAIC (secondary), DIC (fallback).

        [FIX-4.4] PSIS-LOO-CV includes Pareto-k diagnostic.
        When >10% of observations have k > 0.7, the PSIS
        approximation is unreliable and a warning is issued.
        Reference: Vehtari, Gelman & Gabry (2017), Statistics
        and Computing, Section 3.4.

        [FIX-4.5] DIC fallback includes methodological caveat
        that DIC is not uniquely defined for hierarchical models.
        Reference: Gelman, Hwang & Vehtari (2014).
        """
        metrics: Dict[str, Any] = {
            'dic': float('inf'),
            'loo': np.nan,
            'waic': np.nan,
            'loo_reliable': False,
            'pareto_k_pct_bad': np.nan
        }

        # DIC from HDDM model object
        hddm_path = PATHS['models'] / f"hddm_{model_name}.hddm"
        if hddm_path.exists():
            try:
                model = hddm.load(str(hddm_path))
                metrics['dic'] = float(model.dic)
                del model
                gc.collect()
            except Exception as e:
                self.logger.warning(
                    f"  DIC derivation exception: {e}"
                )

        # PSIS-LOO-CV and WAIC from ArviZ InferenceData
        if hasattr(infdata, 'log_likelihood'):
            # --- PSIS-LOO-CV with Pareto-k diagnostic ---
            try:
                loo_result = az.loo(infdata)
                metrics['loo'] = float(loo_result.elpd_loo)

                # [FIX-4.4] Pareto-k diagnostic
                pareto_k = loo_result.pareto_k
                if pareto_k is not None:
                    k_values = np.array(pareto_k).flatten()
                    n_bad = np.sum(k_values > 0.7)
                    pct_bad = float(n_bad / len(k_values) * 100)
                    metrics['pareto_k_pct_bad'] = pct_bad

                    if pct_bad > 10.0:
                        metrics['loo_reliable'] = False
                        self.logger.warning(
                            f"  PSIS-LOO WARNING: "
                            f"{pct_bad:.1f}% of observations "
                            f"have Pareto k > 0.7. "
                            f"PSIS approximation is unreliable. "
                            f"Preferring WAIC/DIC for this model. "
                            f"(Vehtari et al., 2017, Sec 3.4)"
                        )
                    else:
                        metrics['loo_reliable'] = True
                        self.logger.info(
                            f"  PSIS-LOO: elpd={metrics['loo']:.2f}"
                            f", Pareto-k OK "
                            f"({pct_bad:.1f}% > 0.7)"
                        )
                else:
                    # pareto_k not available; assume reliable
                    metrics['loo_reliable'] = True
                    self.logger.info(
                        f"  PSIS-LOO: elpd={metrics['loo']:.2f} "
                        f"(Pareto-k not available)"
                    )

            except Exception as e:
                self.logger.warning(
                    f"  PSIS-LOO derivation exception: {e}"
                )

            # --- WAIC ---
            try:
                waic_result = az.waic(infdata)
                metrics['waic'] = float(waic_result.elpd_waic)
                self.logger.info(
                    f"  WAIC: elpd={metrics['waic']:.2f}"
                )
            except Exception as e:
                self.logger.warning(
                    f"  WAIC derivation exception: {e}"
                )
        else:
            # [FIX-4.5] DIC-only fallback with warning
            self.logger.warning(
                f"  [{model_name}] log_likelihood group absent. "
                f"Only DIC available (no PSIS-LOO/WAIC). "
                f"NOTE: DIC is not uniquely defined for "
                f"hierarchical models and may be unreliable "
                f"(Gelman, Hwang & Vehtari, 2014)."
            )

        return metrics

    # -----------------------------------------------------------------
    # WINNING MODEL DIAGNOSTICS
    # -----------------------------------------------------------------
    def generate_winner_diagnostics(
        self,
        model_name: str,
        infdata: az.InferenceData
    ):
        """
        Constructs comprehensive visualization suite for the
        selected optimal architecture: trace, rank, and pair plots.
        """
        self.logger.info(
            f"\nGenerating diagnostics for optimal architecture: "
            f"[{model_name.upper()}]"
        )

        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            self.logger.warning(
                "Focal parameters absent. "
                "Diagnostics bypassed."
            )
            return

        # 1. Trace plots
        try:
            az.plot_trace(
                infdata,
                var_names=focal_vars,
                compact=True,
                figsize=(12, 2.5 * len(focal_vars))
            )
            plt.savefig(
                PATHS['figures_supp']
                / f"winner_trace_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner trace plot exported.")
        except Exception as e:
            self.logger.warning(
                f"  Trace plot failed: {e}"
            )

        # 2. Rank plots (Vehtari et al., 2021)
        try:
            az.plot_rank(
                infdata,
                var_names=focal_vars,
                kind='vlines',
                vlines_kwargs={'lw': 0},
                marker_vlines_kwargs={'lw': 2}
            )
            plt.savefig(
                PATHS['figures_supp']
                / f"winner_rank_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner rank plot exported.")
        except Exception as e:
            self.logger.warning(
                f"  Rank plot failed: {e}"
            )

        # 3. Posterior pair plots (KDE)
        try:
            pair_vars = (
                focal_vars[:8]
                if len(focal_vars) > 8
                else focal_vars
            )
            az.plot_pair(
                infdata,
                var_names=pair_vars,
                kind='kde',
                marginals=True,
                figsize=(12, 12)
            )
            plt.savefig(
                PATHS['figures_supp']
                / f"winner_pair_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner pair plot exported.")
        except Exception as e:
            self.logger.warning(
                f"  Pair plot failed: {e}"
            )

    # -----------------------------------------------------------------
    # FULL FUNNEL EXECUTION
    # -----------------------------------------------------------------
    def execute_funnel(self):
        """
        Iterates all architectures through the sequential
        four-level evaluation funnel. Populates
        self.comparison_records for downstream model selection.
        """
        for model_name in CFG.final_all_models:
            self.logger.info(f"\n{'─'*50}")
            self.logger.info(
                f"Evaluating: [{model_name.upper()}]"
            )
            self.logger.info(f"{'─'*50}")

            tier = CFG.mcmc_protocols.get(
                model_name, {}
            ).get('tier', 'unknown')

            record = {
                'config_hash': self.active_hash,
                'model_name': model_name,
                'tier': tier,
                'L1_technical': False,
                'L2_converged': False,
                'L3_ppc_adequate': False,
                'dic': float('inf'),
                'loo_elpd': np.nan,
                'loo_reliable': False,
                'pareto_k_pct_bad': np.nan,
                'waic_elpd': np.nan,
                'f_rhat_max': np.nan,
                'f_ess_bulk_min': np.nan,
                'total_kept_samples': np.nan,
                'focal_mcse_mean': np.nan,
                'choice_mae': np.nan,
                'rt_mae': np.nan,
                'overall_pass': False,
                'rejection_reason': 'Pending Evaluation'
            }

            # LEVEL 1
            if not self.evaluate_level1_technical(model_name):
                record['rejection_reason'] = (
                    'L1: Artifact Verification Failed'
                )
                self.comparison_records.append(record)
                self.logger.info(
                    f"  TERMINATED at Level 1 "
                    f"(Artifact Verification)"
                )
                continue

            record['L1_technical'] = True

            nc_path = (
                PATHS['models'] / f"hddm_{model_name}.nc"
            )
            infdata = az.from_netcdf(str(nc_path))

            # LEVEL 2
            l2_result = self.evaluate_level2_convergence(
                infdata, model_name
            )
            record['L2_converged'] = l2_result['pass']
            record['f_rhat_max'] = l2_result.get(
                'f_rhat_max', np.nan
            )
            record['f_ess_bulk_min'] = l2_result.get(
                'f_ess_bulk_min', np.nan
            )

            # Extract total kept samples and focal MCSE for
            # cross-model Monte Carlo quality comparison.
            # These are reported regardless of L2 pass/fail to
            # enable reviewers to assess whether adaptive sampling
            # introduced asymmetric MC noise across models.
            try:
                post = infdata.posterior
                n_chains = post.dims.get('chain', 0)
                n_draws = post.dims.get('draw', 0)
                record['total_kept_samples'] = n_chains * n_draws

                # Focal MCSE: mean of MCSE(mean) across focal params
                summary_tmp = az.summary(
                    infdata, round_to=6, hdi_prob=0.95
                )
                focal_tmp = CFG.identify_focal_parameters(
                    summary_tmp.index.tolist()
                )
                if ('mcse_mean' in summary_tmp.columns
                        and focal_tmp):
                    focal_mcse = summary_tmp.loc[
                        summary_tmp.index.isin(focal_tmp),
                        'mcse_mean'
                    ]
                    record['focal_mcse_mean'] = float(
                        focal_mcse.mean()
                    )
                    self.logger.info(
                        f"  MC quality: "
                        f"{record['total_kept_samples']} kept, "
                        f"focal MCSE(mean)="
                        f"{record['focal_mcse_mean']:.5f}"
                    )
            except Exception as e:
                self.logger.warning(
                    f"  MC quality extraction failed: {e}"
                )

            if not l2_result['pass']:
                record['rejection_reason'] = (
                    f"L2: {l2_result['reason']}"
                )
                self.comparison_records.append(record)
                self.logger.info(
                    f"  TERMINATED at Level 2: "
                    f"{l2_result['reason']}"
                )
                del infdata
                gc.collect()
                continue

            self.logger.info(
                f"  Level 2 Constraints Satisfied"
            )

            # LEVEL 3
            l3_result = self.evaluate_level3_ppc(model_name)
            record['L3_ppc_adequate'] = l3_result['pass']
            record['choice_mae'] = l3_result.get(
                'choice_mae', np.nan
            )
            record['rt_mae'] = l3_result.get('rt_mae', np.nan)

            if not l3_result['pass']:
                record['rejection_reason'] = (
                    f"L3: Predictive Inadequacy "
                    f"(Choice MAE="
                    f"{l3_result['choice_mae']:.4f}, "
                    f"RT MAE={l3_result['rt_mae']:.4f})"
                )
                self.comparison_records.append(record)
                self.logger.info(
                    f"  TERMINATED at Level 3: "
                    f"{record['rejection_reason']}"
                )
                del infdata
                gc.collect()
                continue

            self.logger.info(
                f"  Level 3 Constraints Satisfied "
                f"(Choice MAE="
                f"{l3_result['choice_mae']:.4f})"
            )

            # LEVEL 4
            l4_metrics = self.compute_model_comparison_metrics(
                model_name, infdata
            )
            record['dic'] = l4_metrics['dic']
            record['loo_elpd'] = l4_metrics.get('loo', np.nan)
            record['loo_reliable'] = l4_metrics.get(
                'loo_reliable', False
            )
            record['pareto_k_pct_bad'] = l4_metrics.get(
                'pareto_k_pct_bad', np.nan
            )
            record['waic_elpd'] = l4_metrics.get(
                'waic', np.nan
            )
            record['overall_pass'] = True
            record['rejection_reason'] = (
                'Satisfied Diagnostic Funnel'
            )

            self.comparison_records.append(record)

            self.logger.info(
                f"  Level 4: DIC={l4_metrics['dic']:.2f}, "
                f"LOO elpd="
                f"{l4_metrics.get('loo', 'N/A')}, "
                f"WAIC elpd="
                f"{l4_metrics.get('waic', 'N/A')}"
            )
            self.logger.info(f"  STATUS: FUNNEL COMPLETED")

            del infdata
            gc.collect()

    # -----------------------------------------------------------------
    # OPTIMAL MODEL SELECTION & EXPORT
    # [FIX-4.4] Pareto-k aware selection priority
    # -----------------------------------------------------------------
    def select_optimal_model(self) -> Optional[str]:
        """
        Determines the optimal architecture from the set of
        architectures that satisfied Levels 1-3.

        Selection priority:
          1. PSIS-LOO-CV ELPD (if reliable: Pareto-k < 0.7 for
             >90% of observations)
          2. WAIC ELPD (if LOO unreliable or unavailable)
          3. DIC minimization (structural fallback with caveat)
        """
        df = pd.DataFrame(self.comparison_records)

        if df.empty:
            raise RuntimeError(
                "Funnel execution produced no records. "
                "Ensure execute_funnel() was called first."
            )

        eligible = df[df['overall_pass'] == True].copy()

        if eligible.empty:
            self.logger.error(
                "\nCRITICAL: Zero architectures satisfied "
                "the diagnostic funnel."
            )
            df.to_csv(
                PATHS['tables_main']
                / "model_comparison_summary.csv",
                index=False
            )
            raise RuntimeError(
                "Funnel collapsed. All architectures rejected. "
                "Consult log metrics for adjustments."
            )

        # --- Selection priority logic ---
        # [FIX-4.4] Check LOO reliability via Pareto-k
        has_reliable_loo = (
            'loo_elpd' in eligible.columns
            and 'loo_reliable' in eligible.columns
            and eligible['loo_reliable'].any()
        )

        has_waic = (
            'waic_elpd' in eligible.columns
            and not eligible['waic_elpd'].isna().all()
        )

        if has_reliable_loo:
            # Use only models with reliable LOO
            reliable_mask = eligible['loo_reliable'] == True
            if reliable_mask.any():
                reliable_subset = eligible[reliable_mask]
                winner_idx = reliable_subset['loo_elpd'].idxmax()
                selection_method = "PSIS-LOO-CV (Pareto-k verified)"
            else:
                # All LOO unreliable, fall through to WAIC
                has_reliable_loo = False

        if not has_reliable_loo and has_waic:
            winner_idx = eligible['waic_elpd'].idxmax()
            selection_method = "WAIC (LOO unreliable or unavailable)"
            self.logger.info(
                "  Selection via WAIC (LOO unavailable or "
                "Pareto-k > 0.7 for >10% of observations)."
            )

        if not has_reliable_loo and not has_waic:
            # [FIX-4.5] DIC fallback with caveat
            winner_idx = eligible['dic'].idxmin()
            selection_method = "DIC (fallback; see caveat)"
            self.logger.warning(
                "  Selection via DIC (fallback). "
                "CAVEAT: DIC is not uniquely defined for "
                "hierarchical models and results should be "
                "interpreted with caution. "
                "(Gelman, Hwang & Vehtari, 2014, "
                "Statistics and Computing)"
            )

        winning_model = df.at[winner_idx, 'model_name']
        win_reason = (
            f"OPTIMAL ({selection_method} among "
            f"{len(eligible)} candidates)"
        )

        df['Is_Winner'] = False
        df.at[winner_idx, 'Is_Winner'] = True
        df.at[winner_idx, 'rejection_reason'] = win_reason

        df = df.sort_values(
            by=['overall_pass', 'dic'],
            ascending=[False, True]
        )

        # Export comparison table
        df.to_csv(
            PATHS['tables_main']
            / "model_comparison_summary.csv",
            index=False
        )

        # Export audit record for Steps 5-6
        audit_record = df[df['Is_Winner'] == True].copy()
        audit_record.to_csv(
            PATHS['audit']
            / "final_model_selection_audit.csv",
            index=False
        )

        # Log winner
        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            f"OPTIMAL ARCHITECTURE: [{winning_model.upper()}]"
        )
        self.logger.info(f"  Selection Method: {selection_method}")

        # Report key metric for the winner
        winner_row = df.loc[winner_idx]
        if has_reliable_loo:
            self.logger.info(
                f"  LOO-CV elpd={winner_row['loo_elpd']:.2f}, "
                f"Pareto-k bad="
                f"{winner_row.get('pareto_k_pct_bad', 0):.1f}%"
            )
        elif has_waic:
            self.logger.info(
                f"  WAIC elpd={winner_row['waic_elpd']:.2f}"
            )
        else:
            self.logger.info(
                f"  DIC={winner_row['dic']:.2f}"
            )
        self.logger.info(f"{'='*70}")

        # Consistency check between IC methods
        if (not eligible['loo_elpd'].isna().all()
                and not eligible['dic'].isna().all()):
            dic_best = eligible.loc[
                eligible['dic'].idxmin(), 'model_name'
            ]
            consistent = (dic_best == winning_model)
            self.logger.info(
                f"  DIC Optimum: [{dic_best}] "
                f"({'CONSISTENT' if consistent else 'INCONSISTENT'}"
                f" with primary IC)"
            )

        # Display summary table
        display_cols = [
            'model_name', 'tier',
            'L1_technical', 'L2_converged',
            'L3_ppc_adequate', 'overall_pass',
            'total_kept_samples', 'focal_mcse_mean',
            'loo_elpd', 'loo_reliable',
            'waic_elpd', 'dic',
            'f_rhat_max', 'rejection_reason'
        ]
        available = [c for c in display_cols if c in df.columns]
        print("\n--- Diagnostic Funnel Resolution ---")
        print(df[available].to_string(index=False))

        # Generate winner diagnostics
        nc_path = (
            PATHS['models'] / f"hddm_{winning_model}.nc"
        )
        if nc_path.exists():
            infdata = az.from_netcdf(str(nc_path))
            self.generate_winner_diagnostics(
                winning_model, infdata
            )

            # [FIX-4.6] ArviZ compare with Pareto-k info
            if hasattr(infdata, 'log_likelihood'):
                self._try_arviz_compare(eligible)

            del infdata
            gc.collect()

        return winning_model

    # -----------------------------------------------------------------
    # [FIX-4.6] ARVIZ MODEL COMPARISON TABLE
    # -----------------------------------------------------------------
    def _try_arviz_compare(self, eligible_df: pd.DataFrame):
        """
        Derives formal ArviZ model comparison tables for viable
        models containing valid log_likelihood structures.
        [FIX-4.6] Includes Pareto-k warning information.
        """
        compare_dict = {}
        for _, row in eligible_df.iterrows():
            model_name = row['model_name']
            nc_path = (
                PATHS['models'] / f"hddm_{model_name}.nc"
            )
            if nc_path.exists():
                idata = az.from_netcdf(str(nc_path))
                if hasattr(idata, 'log_likelihood'):
                    compare_dict[model_name] = idata

        if len(compare_dict) < 2:
            for idata in compare_dict.values():
                del idata
            gc.collect()
            return

        try:
            loo_compare = az.compare(compare_dict, ic='loo')
            loo_compare.to_csv(
                PATHS['tables_main']
                / "arviz_loo_comparison.csv"
            )
            self.logger.info(
                "  ArviZ LOO-CV comparison table exported."
            )

            # [FIX-4.6] Check if warning column exists
            if 'warning' in loo_compare.columns:
                warned = loo_compare[
                    loo_compare['warning'] == True
                ]
                if not warned.empty:
                    self.logger.warning(
                        f"  Pareto-k warnings in LOO compare "
                        f"for: {warned.index.tolist()}"
                    )

        except Exception as e:
            self.logger.warning(
                f"  ArviZ LOO comparison exception: {e}"
            )

        try:
            waic_compare = az.compare(compare_dict, ic='waic')
            waic_compare.to_csv(
                PATHS['tables_main']
                / "arviz_waic_comparison.csv"
            )
            self.logger.info(
                "  ArviZ WAIC comparison table exported."
            )
        except Exception as e:
            self.logger.warning(
                f"  ArviZ WAIC comparison exception: {e}"
            )
        finally:
            for idata in compare_dict.values():
                del idata
            gc.collect()


# =============================================================================
# [FIX-4.3] PIPELINE EXECUTION ENTRY POINT
# CRITICAL FIX: execute_funnel() MUST be called before
# select_optimal_model(). The original code skipped this call,
# causing comparison_records to be empty and select_optimal_model()
# to always raise RuntimeError("Funnel execution collapsed...").
# =============================================================================
if __name__ == "__main__":
    funnel = DiagnosticFunnelEngine()

    # [FIX-4.3] Execute the four-level funnel FIRST
    funnel.execute_funnel()

    # THEN select the optimal model from funnel results
    winning_model = funnel.select_optimal_model()

12:02:02 - INFO - Active Pipeline Hash: 719a1af99cc02ea5...
2026-03-29 12:02:02,730 INFO: Active Pipeline Hash: 719a1af99cc02ea5...
12:02:02 - INFO - ======================================================================
2026-03-29 12:02:02,738 INFO: ======================================================================
12:02:02 - INFO - DIAGNOSTIC FUNNEL ENGINE INITIATED (Mode: DEBUG)
2026-03-29 12:02:02,740 INFO: DIAGNOSTIC FUNNEL ENGINE INITIATED (Mode: DEBUG)
12:02:02 - INFO - Pipeline Hash: 719a1af99cc02ea57265605e...
2026-03-29 12:02:02,742 INFO: Pipeline Hash: 719a1af99cc02ea57265605e...
12:02:02 - INFO - Stratified Convergence: Focal R-hat<=1.01, ESS>=1000.0 | Nuisance R-hat<=1.05, ESS>=400.0
2026-03-29 12:02:02,744 INFO: Stratified Convergence: Focal R-hat<=1.01, ESS>=1000.0 | Nuisance R-hat<=1.05, ESS>=400.0
12:02:02 - INFO - PPC Adequacy: Choice MAE<=0.05, RT MAE<=0.05
2026-03-29 12:02:02,746 INFO: PPC Adequacy: Choice MAE<=0.05, RT MAE<=0.05
12:02:02 - INFO - ==============

12:02:20 - INFO -   L2 FOCAL:    R-hat=1.073, ESS_bulk=25, ESS_tail=148 -> FAIL
2026-03-29 12:02:20,879 INFO:   L2 FOCAL:    R-hat=1.073, ESS_bulk=25, ESS_tail=148 -> FAIL
12:02:20 - INFO -   L2 NUISANCE: R-hat=1.119, ESS_bulk=12, ESS_tail=14 -> FAIL
2026-03-29 12:02:20,881 INFO:   L2 NUISANCE: R-hat=1.119, ESS_bulk=12, ESS_tail=14 -> FAIL
12:02:21 - INFO -   MC quality: 1000 kept, focal MCSE(mean)=0.00490
2026-03-29 12:02:21,462 INFO:   MC quality: 1000 kept, focal MCSE(mean)=0.00490
12:02:21 - INFO -   TERMINATED at Level 2: Focal: R-hat=1.073,ESS_b=25,ESS_t=148 | Nuisance: R-hat=1.119,ESS_b=12
2026-03-29 12:02:21,464 INFO:   TERMINATED at Level 2: Focal: R-hat=1.073,ESS_b=25,ESS_t=148 | Nuisance: R-hat=1.119,ESS_b=12
12:02:21 - INFO - 
──────────────────────────────────────────────────
2026-03-29 12:02:21,587 INFO: 
──────────────────────────────────────────────────
12:02:21 - INFO - Evaluating: [VAZT]
2026-03-29 12:02:21,589 INFO: Evaluating: [VAZT]
12:02:21 - INFO - ──────────────

RuntimeError: Funnel collapsed. All architectures rejected. Consult log metrics for adjustments.

# Step 5: Statistical Inference and Publication-Ready Visualization

In [6]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 5: Bayesian Posterior Inference, HDI/ROPE Evaluation &
        Publication Visualization (Physical Scale)
=============================================================================
Pipeline Position:
  Upstream:   Step 4 (reads final_model_selection_audit.csv to identify
              winning model; reads .nc file from models/ directory)
  Downstream: Step 6 (no direct downstream dependency; this is the
              primary inference output step)

Methodological Purpose:
  - Retrieves the optimal architecture's InferenceData from Step 4.
  - Enforces cryptographic lineage validation.
  - CRITICAL: Reconstructs absolute posterior distributions and
    treatment contrasts via Inverse Link Functions (exp for 'a'/'t',
    expit for 'z') PRIOR to inferential computation. This ensures
    HDI and ROPE are evaluated on ecologically valid physical scales.
  - Computes rigorous Bayesian inferential metrics per physical contrast:
    * 95% Highest Density Interval (HDI)
    * Probability of Direction (Pd; Makowski et al., 2019)
    * ROPE overlap (Kruschke, 2018)
    * HDI-zero exclusion
  - Constructs publication-ready visualizations:
    * ArviZ az.plot_posterior() with physical scale ROPE annotations.
    * Raincloud/violin plots for absolute posteriors.
    * Contrast KDE density plots with HDI bars and ROPE shading.

=============================================================================
"""

import os
import gc
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.special import expit

# Suppress inconsequential dependency warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# -----------------------------------------------------------------------------
# [FIX-5.1] CONFIGURATION IMPORT
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state,
        identify_winning_model
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' unresolved. "
        "Execution of Step 2a is mandatory."
    )

PATHS = CFG.initialize_directories()

# [FIX-5.2] Deterministic seed from configuration
np.random.seed(CFG.base_seed)

# Publication-grade aesthetics
sns.set_theme(style="ticks", palette="colorblind")
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
    'font.size': 12,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.labelsize': 14,
    'legend.fontsize': 11,
    'legend.frameon': False,
    'figure.autolayout': True
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_inference_logger() -> logging.Logger:
    """Initializes dual-sink logging for statistical inference."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_inference_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'statistical_inference_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# CORE CLASS: BAYESIAN INFERENCE & VISUALIZATION ENGINE
# =============================================================================
class BayesianInferenceVisualizer:
    """
    Orchestrates posterior extraction, inverse-link transformations,
    HDI/ROPE hypothesis testing on the physical scale, and
    publication-quality visualization for the winning HDDM model.
    """

    def __init__(self):
        self.logger = _setup_inference_logger()

        # [FIX-5.1] Lineage validation
        self.active_lineage = load_active_lineage_state(
            PATHS, self.logger
        )
        self.active_hash = self.active_lineage['pipeline_hash']

        self.winning_model = identify_winning_model(PATHS)
        self.infdata: Optional[az.InferenceData] = None
        self.stats_records: List[Dict] = []

        # Cache physical contrasts for visualization
        self.physical_contrasts_dict: Dict[str, np.ndarray] = {}

        # Experimental design variables from SSOT
        self.baseline = CFG.baseline_condition
        self.emotions = CFG.emotion_order
        self.colors = CFG.colors
        self.labels = CFG.display_labels

        self.logger.info("=" * 70)
        self.logger.info(
            f"BAYESIAN INFERENCE ENGINE INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(
            f"Pipeline Hash: {self.active_hash[:24]}..."
        )
        self.logger.info(
            f"Target Architecture: "
            f"[{self.winning_model.upper()}]"
        )
        self.logger.info("=" * 70)

        self._load_inference_data()

    def _load_inference_data(self):
        """Loads the winning model's InferenceData from NetCDF."""
        nc_path = (
            PATHS['models'] / f"hddm_{self.winning_model}.nc"
        )
        if not nc_path.exists():
            raise FileNotFoundError(
                f"InferenceData unresolved: {nc_path.name}. "
                f"Ensure Steps 2b-4 completed."
            )
        self.infdata = az.from_netcdf(str(nc_path))
        self.logger.info(
            f"InferenceData imported: "
            f"groups={list(self.infdata.groups())}"
        )

    # -----------------------------------------------------------------
    # ROPE BOUNDARY DEFINITIONS (PHYSICAL SCALE)
    # -----------------------------------------------------------------
    @staticmethod
    def _define_rope_for_param(
        param_family: str
    ) -> Tuple[float, float]:
        """
        Establishes ROPE boundaries per DDM parameter family.
        Bounds operate on the PHYSICAL scale (absolute shift).

        v: drift rate shift (identity link, units: evidence/sec)
        a: boundary separation shift (exp link, units: arb.)
        t: non-decision time shift (exp link, units: seconds)
        z: starting point shift (logit link, units: probability)
        """
        rope_mapping = {
            'v': (-0.20, 0.20),
            'a': (-0.05, 0.05),
            't': (-0.01, 0.01),
            'z': (-0.02, 0.02)
        }
        return rope_mapping.get(param_family, (-0.10, 0.10))

    # -----------------------------------------------------------------
    # INVERSE LINK FUNCTION
    # -----------------------------------------------------------------
    @staticmethod
    def _apply_inverse_link(
        param_family: str, trace: np.ndarray
    ) -> np.ndarray:
        """
        Applies HDDM internal inverse link transformations to
        linear predictor traces, converting to the physical scale.

        HDDM internal parameterization:
          v: identity (no transformation)
          a: exp(linear_predictor), clipped for numerical stability
          t: exp(linear_predictor), clipped for numerical stability
          z: expit(linear_predictor) = 1/(1+exp(-lp))
        """
        if param_family == 'v':
            return trace
        elif param_family == 'a':
            return np.exp(np.clip(trace, -10.0, 4.0))
        elif param_family == 't':
            return np.exp(np.clip(trace, -10.0, 2.0))
        elif param_family == 'z':
            return expit(trace)
        return trace

    # -----------------------------------------------------------------
    # [FIX-5.5] POSTERIOR TRACE EXTRACTION WITH DYNAMIC DETECTION
    # -----------------------------------------------------------------
    def _get_flattened_traces(self) -> pd.DataFrame:
        """
        Extracts posterior from InferenceData, flattening the
        (chain, draw) tensor into a single sample axis.
        """
        post = self.infdata.posterior
        traces = {}
        for var_name in post.data_vars:
            vals = post[var_name].values
            # Handle multi-dimensional variables (e.g., subject-level)
            if vals.ndim > 2:
                # Skip subject-level parameters; only extract
                # group-level (scalar per chain x draw)
                continue
            traces[var_name] = vals.flatten()
        return pd.DataFrame(traces)

    def _find_trace_column(
        self, traces: pd.DataFrame, candidates: List[str]
    ) -> Optional[str]:
        """
        [FIX-5.5] Dynamic column detection for group-level parameters.

        When group_only_regressors=False, HDDM may suffix group-level
        parameters differently depending on version. This method
        tries a prioritized list of candidate column names and returns
        the first match found.
        """
        for candidate in candidates:
            if candidate in traces.columns:
                return candidate
        return None

    # -----------------------------------------------------------------
    # RECONSTRUCTION & INFERENTIAL COMPUTATION
    # -----------------------------------------------------------------
    def reconstruct_and_evaluate(
        self, param_family: str
    ) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Reconstructs absolute posterior densities and evaluates
        contrast HDI/ROPE for experimental conditions within a
        specified DDM parameter family.

        Transformation Logic:
          1. Base_LP = Intercept (linear predictor scale)
          2. Base_Physical = InverseLink(Base_LP)
          3. Cond_LP = Intercept + TreatmentContrast
          4. Cond_Physical = InverseLink(Cond_LP)
          5. Physical_Contrast = Cond_Physical - Base_Physical
          6. HDI/ROPE evaluated on Physical_Contrast

        [FIX-5.5] Intercept and contrast column names are detected
        dynamically to accommodate both group_only_regressors=True
        and False configurations.
        """
        traces = self._get_flattened_traces()
        reconstructed_data = []
        inference_records = []

        # [FIX-5.5] Dynamic intercept detection
        intercept_candidates = [
            f"{param_family}_Intercept",
            f"{param_family}",
            f"{param_family}_mu_Intercept",
        ]
        intercept_col = self._find_trace_column(
            traces, intercept_candidates
        )
        if intercept_col is None:
            self.logger.warning(
                f"Intercept for '{param_family}' unresolved. "
                f"Tried: {intercept_candidates}. "
                f"Available columns: "
                f"{[c for c in traces.columns if c.startswith(param_family)]}"
            )
            return pd.DataFrame(), []

        self.logger.info(
            f"  [{param_family.upper()}] Intercept column: "
            f"'{intercept_col}'"
        )

        base_lp_trace = traces[intercept_col].values
        base_phys_trace = self._apply_inverse_link(
            param_family, base_lp_trace
        )

        reconstructed_data.append(pd.DataFrame({
            "Emotion": self.baseline,
            "Posterior_Value": base_phys_trace
        }))

        rope_bounds = self._define_rope_for_param(param_family)

        for em in self.emotions:
            if em == self.baseline:
                continue

            # [FIX-5.5] Dynamic contrast column detection
            contrast_candidates = [
                (
                    f"{param_family}_C(emotion, "
                    f"Treatment('{self.baseline}'))[T.{em}]"
                ),
                f"{param_family}_C(emotion)[T.{em}]",
                (
                    f"{param_family}_C(emotion, "
                    f"Treatment(reference='{self.baseline}'))"
                    f"[T.{em}]"
                ),
            ]
            contrast_col = self._find_trace_column(
                traces, contrast_candidates
            )

            if contrast_col is None:
                self.logger.warning(
                    f"  Contrast unresolved for "
                    f"{param_family} x {em}. "
                    f"Tried: {contrast_candidates[:2]}..."
                )
                continue

            contrast_lp_trace = traces[contrast_col].values
            cond_lp_trace = base_lp_trace + contrast_lp_trace
            cond_phys_trace = self._apply_inverse_link(
                param_family, cond_lp_trace
            )

            # Physical contrast magnitude
            phys_contrast_trace = (
                cond_phys_trace - base_phys_trace
            )

            safe_var_name = (
                f"{param_family}_{em}_vs_{self.baseline}"
            )
            self.physical_contrasts_dict[safe_var_name] = (
                phys_contrast_trace
            )

            # --- Bayesian Inferential Computation ---
            hdi_bounds = az.hdi(
                phys_contrast_trace, hdi_prob=0.95
            )
            median_diff = float(np.median(phys_contrast_trace))
            mean_diff = float(np.mean(phys_contrast_trace))

            p_greater = float(
                np.mean(phys_contrast_trace > 0)
            )
            pd_val = max(p_greater, 1.0 - p_greater)

            rope_overlap = float(np.mean(
                (phys_contrast_trace >= rope_bounds[0])
                & (phys_contrast_trace <= rope_bounds[1])
            ))

            hdi_excludes_zero = (
                (hdi_bounds[0] > 0) or (hdi_bounds[1] < 0)
            )

            inference_records.append({
                'config_hash': (
                    self.active_lineage['config_hash']
                ),
                'pipeline_hash': self.active_hash,
                'model_name': self.winning_model,
                'Parameter_Family': param_family,
                'Condition': em,
                'Contrast_vs': self.baseline,
                'Physical_Contrast_Mean': mean_diff,
                'Physical_Contrast_Median': median_diff,
                'HDI_95_Low': float(hdi_bounds[0]),
                'HDI_95_High': float(hdi_bounds[1]),
                'HDI_Excludes_Zero': hdi_excludes_zero,
                'Pd': pd_val,
                'P_Greater_Than_Zero': p_greater,
                'ROPE_Low': rope_bounds[0],
                'ROPE_High': rope_bounds[1],
                'ROPE_Overlap_Pct': rope_overlap * 100
            })

            reconstructed_data.append(pd.DataFrame({
                "Emotion": em,
                "Posterior_Value": cond_phys_trace
            }))

        if reconstructed_data:
            return (
                pd.concat(reconstructed_data, ignore_index=True),
                inference_records
            )
        return pd.DataFrame(), inference_records

    def compute_all_posterior_statistics(self):
        """
        Iterates over parameter families parameterized in the
        optimal architecture, computing full inferential matrices.
        """
        self.logger.info(
            "\nComputing Bayesian HDI/ROPE metrics "
            "(Physical Scale)..."
        )

        model_lower = self.winning_model.lower()
        families = [
            p for p in ['v', 'a', 't', 'z']
            if p in model_lower or model_lower == 'null'
        ]

        # Null model has all families as intercept-only
        if model_lower == 'null':
            self.logger.info(
                "  Null model detected. No treatment contrasts "
                "to evaluate."
            )
            return

        all_records = []
        for family in families:
            _, records = self.reconstruct_and_evaluate(family)
            if records:
                all_records.extend(records)
                self.logger.info(
                    f"  [{family.upper()}] "
                    f"{len(records)} contrasts evaluated."
                )

        self.stats_records = all_records
        df_stats = pd.DataFrame(all_records)

        if not df_stats.empty:
            out_path = (
                PATHS['tables_main']
                / f"bayesian_inference_physical_summary_"
                  f"{self.winning_model}.csv"
            )
            df_stats.to_csv(out_path, index=False)
            self.logger.info(
                f"Inferential matrix exported: {out_path.name} "
                f"({len(df_stats)} records)"
            )

            print(
                "\n--- Bayesian Inference Summary "
                "(Physical Scale) ---"
            )
            display_cols = [
                'Parameter_Family', 'Condition',
                'Physical_Contrast_Median',
                'HDI_95_Low', 'HDI_95_High',
                'HDI_Excludes_Zero',
                'Pd', 'ROPE_Overlap_Pct'
            ]
            print(df_stats[display_cols].to_string(index=False))
        else:
            self.logger.warning(
                "No valid contrast structures identified."
            )

    # -----------------------------------------------------------------
    # VISUALIZATION 1: ArviZ Posterior + ROPE Plot
    # -----------------------------------------------------------------
    def render_arviz_posterior_rope(self):
        """
        Deploys az.plot_posterior() with ROPE annotations on
        PHYSICAL contrasts. Constructs a pseudo-InferenceData
        from derived physical disparities.
        """
        self.logger.info(
            "\nRendering ArviZ posterior + ROPE "
            "(Physical Scale)..."
        )

        if not self.physical_contrasts_dict:
            self.logger.warning(
                "Physical contrast mapping absent. "
                "Visualization bypassed."
            )
            return

        # Build pseudo (chain, draw) structure for ArviZ
        mock_posteriors = {}
        for var_name, trace in (
            self.physical_contrasts_dict.items()
        ):
            mock_posteriors[var_name] = np.expand_dims(
                trace, axis=0
            )

        mock_infdata = az.from_dict(posterior=mock_posteriors)

        for var_name in mock_posteriors.keys():
            family = var_name.split('_')[0]
            rope = self._define_rope_for_param(family)

            try:
                az.plot_posterior(
                    mock_infdata,
                    var_names=[var_name],
                    hdi_prob=0.95,
                    rope=rope,
                    figsize=(8, 4),
                    textsize=12
                )
                plt.title(
                    f"Physical Contrast: {var_name}",
                    fontsize=14, pad=10
                )
                plt.tight_layout()

                out_path = (
                    PATHS['figures_main']
                    / f"posterior_rope_physical_{var_name}.pdf"
                )
                plt.savefig(
                    out_path, dpi=300, bbox_inches='tight'
                )
                plt.close()
                self.logger.info(
                    f"  Exported: {out_path.name}"
                )
            except Exception as e:
                self.logger.warning(
                    f"  az.plot_posterior failed for "
                    f"{var_name}: {e}"
                )

    # -----------------------------------------------------------------
    # [FIX-5.3] VISUALIZATION 2: Violin + Boxplot (Absolute)
    # -----------------------------------------------------------------
    def render_violin_absolute(self, param_family: str):
        """
        Constructs violin + boxplot distributions depicting
        reconstructed absolute posterior mass (Physical Scale)
        segmented by condition.

        [FIX-5.3] Replaced Raincloud plot with violin + boxplot.
        The original Raincloud used all MCMC samples as jitter
        points, which misleadingly resembled participant-level
        observations. Allen et al. (2019, Wellcome Open Research)
        specifies that the scatter component should show OBSERVED
        data points (e.g., per-subject means), not MCMC draws.
        Since we are plotting posterior distributions (not observed
        data), violin + boxplot is the appropriate representation.
        """
        df_recon, _ = self.reconstruct_and_evaluate(param_family)
        if df_recon.empty:
            return

        df_recon["Emotion_Label"] = df_recon["Emotion"].map(
            self.labels
        )
        ordered_labels = [
            self.labels[em] for em in self.emotions
            if em in df_recon["Emotion"].unique()
        ]
        ordered_colors = [
            self.colors.get(em, '#000000')
            for em in self.emotions
            if em in df_recon["Emotion"].unique()
        ]

        # Subsample for plotting efficiency
        # (full MCMC trace can have 20k+ samples per condition)
        max_plot_samples = 5000
        df_plot = df_recon.groupby('Emotion').apply(
            lambda x: x.sample(
                n=min(len(x), max_plot_samples),
                random_state=CFG.base_seed
            )
        ).reset_index(drop=True)

        fig, ax = plt.subplots(figsize=(10, 6))

        sns.violinplot(
            x="Emotion_Label", y="Posterior_Value",
            data=df_plot,
            palette=ordered_colors, order=ordered_labels,
            inner=None, cut=0, ax=ax, alpha=0.5,
            linewidth=1.5
        )

        sns.boxplot(
            x="Emotion_Label", y="Posterior_Value",
            data=df_plot,
            palette=ordered_colors, order=ordered_labels,
            width=0.15, ax=ax, fliersize=0,
            boxprops=dict(alpha=0.8),
            medianprops=dict(color='black', linewidth=2),
            whiskerprops=dict(linewidth=1.5),
            capprops=dict(linewidth=1.5)
        )

        # Baseline median reference line
        baseline_median = df_recon[
            df_recon["Emotion"] == self.baseline
        ]["Posterior_Value"].median()
        ax.axhline(
            baseline_median, color='gray', linestyle='--',
            alpha=0.5, zorder=0,
            label=f'{self.labels[self.baseline]} Median'
        )

        # HDI / ROPE annotations
        family_records = [
            r for r in self.stats_records
            if r['Parameter_Family'] == param_family
        ]
        for record in family_records:
            cond_label = self.labels.get(
                record['Condition'], ''
            )
            if cond_label not in ordered_labels:
                continue
            idx = ordered_labels.index(cond_label)

            hdi_low = record['HDI_95_Low']
            hdi_high = record['HDI_95_High']
            rope_pct = record['ROPE_Overlap_Pct']

            cond_data = df_recon[
                df_recon["Emotion"] == record['Condition']
            ]["Posterior_Value"]
            y_top = (
                cond_data.quantile(0.975)
                + 0.05 * abs(cond_data.std())
            )

            annotation = (
                f"HDI: [{hdi_low:.3f}, {hdi_high:.3f}]\n"
                f"ROPE: {rope_pct:.1f}%"
            )
            ax.annotate(
                annotation, xy=(idx, y_top),
                fontsize=8, ha='center', va='bottom',
                bbox=dict(
                    boxstyle='round,pad=0.3',
                    facecolor='white',
                    edgecolor='gray', alpha=0.8
                )
            )

        ax.set_title(
            f"Absolute Posterior: {param_family.upper()} "
            f"({self.winning_model.upper()})",
            fontsize=16, fontweight='bold', pad=20
        )
        ax.set_ylabel(
            f"Physical Parameter Value ({param_family})",
            fontsize=14
        )
        ax.set_xlabel("Experimental Condition", fontsize=14)
        ax.legend(fontsize=10, loc='best')
        sns.despine(trim=True)

        fig.tight_layout()
        fig.savefig(
            PATHS['figures_main']
            / f"posterior_violin_physical_"
              f"{param_family}_{self.winning_model}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)
        self.logger.info(
            f"  Violin plot exported: "
            f"posterior_violin_physical_"
            f"{param_family}_{self.winning_model}.pdf"
        )

    # -----------------------------------------------------------------
    # [FIX-5.4] VISUALIZATION 3: Contrast KDE with ROPE Shading
    # -----------------------------------------------------------------
    def render_contrast_kde(self):
        """
        Constructs overlapping KDE for Physical Treatment Effects
        annotated with HDI bars and ROPE shading regions.

        [FIX-5.4] Added ROPE shading as semi-transparent grey
        rectangle overlaid on the KDE plot. This allows readers
        to visually assess what proportion of the posterior
        distribution falls within the region of practical
        equivalence (Kruschke, 2018).
        """
        self.logger.info(
            "\nRendering contrast KDE (Physical Scale)..."
        )

        model_lower = self.winning_model.lower()
        families = [
            p for p in ['v', 'a', 't', 'z']
            if p in model_lower
        ]

        for family in families:
            family_records = [
                r for r in self.stats_records
                if r['Parameter_Family'] == family
            ]
            if not family_records:
                continue

            rope_bounds = self._define_rope_for_param(family)

            fig, ax = plt.subplots(figsize=(8, 5))

            # Null effect reference line
            ax.axvline(
                x=0, color='black', linestyle='--',
                linewidth=1.5, zorder=1, label="Null Effect"
            )

            # [FIX-5.4] ROPE shading
            ax.axvspan(
                rope_bounds[0], rope_bounds[1],
                alpha=0.12, color='gray', zorder=0,
                label=(
                    f"ROPE [{rope_bounds[0]}, "
                    f"{rope_bounds[1]}]"
                )
            )

            y_offset = 0.0
            for record in family_records:
                cond = record['Condition']
                if cond not in self.colors:
                    continue

                var_name = (
                    f"{family}_{cond}_vs_{self.baseline}"
                )
                if var_name not in self.physical_contrasts_dict:
                    continue

                contrast_vals = (
                    self.physical_contrasts_dict[var_name]
                )
                color = self.colors[cond]
                label = self.labels.get(cond, cond)

                sns.kdeplot(
                    contrast_vals, fill=True, color=color,
                    alpha=0.4, linewidth=2, label=label,
                    ax=ax, zorder=3
                )

                # HDI bar below x-axis
                hdi_low = record['HDI_95_Low']
                hdi_high = record['HDI_95_High']
                y_offset -= 0.15
                ax.plot(
                    [hdi_low, hdi_high],
                    [y_offset, y_offset],
                    color=color, linewidth=4,
                    solid_capstyle='round', zorder=4
                )
                ax.plot(
                    [record['Physical_Contrast_Median']],
                    [y_offset],
                    marker='|', color=color,
                    markersize=12, markeredgewidth=2,
                    zorder=5
                )

            ax.set_title(
                f"Physical Treatment Effects: "
                f"{family.upper()} vs. "
                f"{self.labels[self.baseline]}",
                fontsize=14, fontweight='bold'
            )
            ax.set_xlabel(
                f"Physical Shift in {family.upper()} "
                f"(\u0394 from baseline)",
                fontsize=12
            )
            ax.set_ylabel("Density", fontsize=12)
            ax.legend(loc='upper right', fontsize=10)
            sns.despine(trim=True)

            fig.tight_layout()
            fig.savefig(
                PATHS['figures_supp']
                / f"contrast_kde_physical_"
                  f"{family}_{self.winning_model}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close(fig)
            self.logger.info(
                f"  Contrast KDE exported: "
                f"contrast_kde_physical_"
                f"{family}_{self.winning_model}.pdf"
            )

    # -----------------------------------------------------------------
    # MASTER PIPELINE
    # -----------------------------------------------------------------
    def run(self):
        """Orchestrates sequential inference and visualization."""
        self.compute_all_posterior_statistics()
        self.render_arviz_posterior_rope()

        model_lower = self.winning_model.lower()
        families = [
            p for p in ['v', 'a', 't', 'z']
            if p in model_lower
        ]
        for family in families:
            self.render_violin_absolute(family)

        self.render_contrast_kde()

        # Memory cleanup
        del self.infdata
        self.infdata = None
        self.physical_contrasts_dict.clear()
        gc.collect()

        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            f"STEP 5 TERMINATED: Inference completed for "
            f"[{self.winning_model.upper()}]."
        )
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    engine = BayesianInferenceVisualizer()
    engine.run()

12:03:42 - INFO - Active Pipeline Hash: 719a1af99cc02ea5...
2026-03-29 12:03:42,954 INFO: Active Pipeline Hash: 719a1af99cc02ea5...


FileNotFoundError: No audit manifest found. Ensure Step 4 completed successfully.

# Step 6: Data Informed Group-Level Parameter Recovery

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 6: Multi-Iteration Dual-Track Parameter Recovery Validation
=============================================================================
Pipeline Position:
  Upstream:   Step 4 (reads final_model_selection_audit.csv to identify
              winning model; reads .nc file from models/)
              Step 1 (reads hddm_data_unfair.csv for design skeleton)
  Downstream: None (terminal validation step)

Methodological Purpose:
  - Validates the structural identifiability of the optimal model
    via two complementary forward-simulation-and-refitting paradigms:
    1. Posterior-Anchored: Extracts pseudo-truth from the empirical
       joint posterior to evaluate self-consistency.
    2. Prior-Predictive: Samples pseudo-truth from ecologically valid
       informative priors to stress-test identifiability.
  - Executes N_ITERATIONS (default 50) independent recovery cycles
    per mode to obtain distributional estimates of Bias, RMSE,
    Coverage Rate, and Correlation (Wilson & Collins, 2019, eLife).
  - Constructs synthetic datasets preserving the empirical trial
    skeleton WITH subject-level parameter variability drawn from
    the hierarchical prior structure (Lerche & Voss, 2016).
  - Validates refit convergence before computing recovery metrics.
  - Implements per-iteration timeout, memory guard, and incremental
    result persistence to survive interruptions.
=============================================================================
"""

import os
import gc
import time
import logging
import traceback
import signal
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import arviz as az
import hddm
from hddm.generate import gen_rand_data

from scipy.special import expit, logit
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# -----------------------------------------------------------------------------
# [FIX-6.1] CONFIGURATION IMPORT
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state,
        identify_winning_model,
        check_memory_headroom
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' unresolved. "
        "Execution of Step 2a is mandatory."
    )

PATHS = CFG.initialize_directories()

# [FIX-6.1] Deterministic seed from configuration
np.random.seed(CFG.base_seed)

# Publication-grade aesthetics
sns.set_theme(style="ticks", palette="colorblind")
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False,
    'figure.autolayout': True
})

# =============================================================================
# RECOVERY CONFIGURATION CONSTANTS
# =============================================================================
# [FIX-6.2] Number of independent recovery iterations per mode
N_ITERATIONS_FINAL = 50
N_ITERATIONS_DEBUG = 3

# [FIX-6.5] Safety limits
ITERATION_TIMEOUT_SEC = 2700  # 45 minutes per iteration
MAX_CONSECUTIVE_FAILURES = 5

# [FIX-6.4] Convergence threshold for recovery refits
RECOVERY_RHAT_THRESHOLD = 1.10


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_recovery_logger() -> logging.Logger:
    """Initializes dual-sink logging for recovery module."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_recovery_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'dual_recovery_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# TIMEOUT HANDLER (POSIX only; graceful no-op on Windows)
# =============================================================================
class TimeoutError(Exception):
    """Custom timeout exception for recovery iteration guard."""
    pass


def _timeout_handler(signum, frame):
    raise TimeoutError("Recovery iteration exceeded time limit.")


def _set_timeout(seconds: int):
    """Sets alarm-based timeout. No-op on Windows."""
    if hasattr(signal, 'SIGALRM'):
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.alarm(seconds)


def _clear_timeout():
    """Clears pending alarm. No-op on Windows."""
    if hasattr(signal, 'SIGALRM'):
        signal.alarm(0)


# =============================================================================
# DYNAMIC REGRESSOR FACTORY
# =============================================================================
def _build_regressors(
    model_name: str, baseline: str
) -> List[str]:
    """
    Constructs Patsy formulas matching Step 2b specifications.
    ALL 4 DDM parameters get explicit formulas (treatment-coded
    or intercept-only) to ensure dockerHDDM compatibility.
    """
    name_lower = model_name.lower()
    regressors = []

    for param in ['v', 'a', 't', 'z']:
        if name_lower != 'null' and param in name_lower:
            regressors.append(
                f"{param} ~ C(emotion, "
                f"Treatment('{baseline}'))"
            )
        else:
            regressors.append(f"{param} ~ 1")

    return regressors


# =============================================================================
# CORE CLASS: MULTI-ITERATION DUAL-TRACK RECOVERY ENGINE
# =============================================================================
class DualParameterRecoveryEngine:
    """
    Orchestrates N-iteration Posterior-Anchored and Prior-Predictive
    identifiability evaluations for the winning HDDM architecture.
    """

    def __init__(
        self,
        empirical_data_path: str = 'hddm_data_unfair.csv'
    ):
        self.logger = _setup_recovery_logger()

        self.active_lineage = load_active_lineage_state(
            PATHS, self.logger
        )
        self.active_hash = (
            self.active_lineage['pipeline_hash']
        )
        self.winning_model = identify_winning_model(PATHS)
        self.empirical_data_path = empirical_data_path

        # Recovery MCMC hyperparameters (lighter than main fitting)
        if CFG.run_mode == 'debug':
            self.recovery_chains = 2
            self.recovery_samples = 500
            self.recovery_burn = 100
            self.n_iterations = N_ITERATIONS_DEBUG
        else:
            self.recovery_chains = CFG.n_chains
            self.recovery_samples = 3000
            self.recovery_burn = 1000
            self.n_iterations = N_ITERATIONS_FINAL

        # [FIX-6.3] Load empirical data for design skeleton
        self.empirical_df = pd.read_csv(empirical_data_path)
        self.empirical_df['subj_idx'] = (
            self.empirical_df['subj_idx'].astype(str)
        )
        self.n_subjects = self.empirical_df['subj_idx'].nunique()

        self.logger.info("=" * 70)
        self.logger.info(
            f"MULTI-ITERATION RECOVERY INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(
            f"Pipeline Hash: {self.active_hash[:24]}..."
        )
        self.logger.info(
            f"Target: [{self.winning_model.upper()}]"
        )
        self.logger.info(
            f"Iterations: {self.n_iterations} per mode"
        )
        self.logger.info(
            f"Recovery MCMC: {self.recovery_samples} samples, "
            f"{self.recovery_burn} burn, "
            f"{self.recovery_chains} chains"
        )
        self.logger.info(
            f"Design: {self.n_subjects} subjects"
        )
        self.logger.info("=" * 70)

    # -----------------------------------------------------------------
    # LINK FUNCTIONS
    # -----------------------------------------------------------------
    @staticmethod
    def _apply_link_and_clip(
        param_family: str, lp_val: float
    ) -> float:
        """
        Applies HDDM inverse link and enforces physiological bounds.
        v: identity, clipped to [-8, 8]
        a: exp, clipped to [0.2, 4.0]
        t: exp, clipped to [0.05, 2.0]
        z: expit, clipped to [0.05, 0.95]
        """
        if param_family == 'v':
            return float(np.clip(lp_val, -8.0, 8.0))
        elif param_family == 'a':
            return float(np.clip(np.exp(lp_val), 0.2, 4.0))
        elif param_family == 't':
            return float(np.clip(np.exp(lp_val), 0.05, 2.0))
        elif param_family == 'z':
            return float(np.clip(expit(lp_val), 0.05, 0.95))
        return lp_val

    # -----------------------------------------------------------------
    # GROUND TRUTH ESTABLISHMENT
    # [FIX-6.3] Includes subject-level variability
    # -----------------------------------------------------------------
    def establish_ground_truth(
        self, mode: str, iteration: int
    ) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
        """
        Synthesizes pseudo-truth parameter matrices.

        Returns:
          group_truth: Dict of group-level LP-scale parameters.
          subject_truths: Dict mapping subj_idx -> dict of
                         physical-scale parameters per condition.
                         Includes subject-level variability drawn
                         from hierarchical prior structure.

        [FIX-6.3] Subject-level parameters are sampled as:
          beta_i ~ N(mu_beta, sigma_beta)
        where sigma_beta is a fixed fraction of the parameter
        range, preserving realistic inter-individual differences.
        """
        # Unique seed per mode x iteration
        iter_seed = (
            CFG.base_seed + iteration * 100
            + (0 if mode == 'posterior' else 50)
        )
        np.random.seed(iter_seed)

        group_truth = {}
        model_lower = self.winning_model.lower()

        if mode == 'posterior':
            nc_path = (
                PATHS['models']
                / f"hddm_{self.winning_model}.nc"
            )
            if not nc_path.exists():
                raise FileNotFoundError(
                    f"InferenceData unresolved: {nc_path.name}"
                )

            infdata = az.from_netcdf(str(nc_path))
            summary = az.summary(infdata, round_to=4)

            # Extract focal fixed-effect parameters
            focal_indices = [
                idx for idx in summary.index
                if ('Intercept' in idx or 'Treatment' in idx
                    or 'C(emotion' in idx)
                and not any(
                    s in idx for s in
                    ['_subj', '_std', '_var', '_log', '_trans']
                )
            ]
            group_truth = (
                summary.loc[focal_indices, 'mean'].to_dict()
            )

            # For posterior mode, sample FROM posterior draws
            # rather than using the mean (more realistic)
            if iteration > 0:
                post = infdata.posterior
                draw_idx = np.random.randint(
                    0, post.dims['draw']
                )
                chain_idx = np.random.randint(
                    0, post.dims['chain']
                )
                for param in focal_indices:
                    if param in post.data_vars:
                        val = float(
                            post[param]
                            .values[chain_idx, draw_idx]
                        )
                        group_truth[param] = val

            del infdata
            gc.collect()

        elif mode == 'prior':
            for param_family in ['v', 'a', 't', 'z']:
                is_varying = (
                    param_family in model_lower
                    and model_lower != 'null'
                )

                # Intercept synthesis
                if param_family == 'v':
                    phys_val = stats.norm.rvs(
                        loc=1.0, scale=1.5
                    )
                    lp_int = phys_val
                elif param_family == 'a':
                    phys_val = stats.gamma.rvs(
                        a=9.0, scale=0.166
                    )
                    phys_val = np.clip(phys_val, 0.5, 3.5)
                    lp_int = np.log(phys_val)
                elif param_family == 't':
                    phys_val = stats.truncnorm.rvs(
                        a=(0.1 - 0.3) / 0.1,
                        b=(0.5 - 0.3) / 0.1,
                        loc=0.3, scale=0.1
                    )
                    phys_val = np.clip(phys_val, 0.05, 0.8)
                    lp_int = np.log(phys_val)
                elif param_family == 'z':
                    phys_val = stats.beta.rvs(a=5, b=5)
                    phys_val = np.clip(phys_val, 0.1, 0.9)
                    lp_int = logit(phys_val)
                else:
                    lp_int = 0.0

                group_truth[
                    f"{param_family}_Intercept"
                ] = float(lp_int)

                # Treatment contrasts
                if is_varying:
                    for emo in CFG.emotion_order:
                        if emo == CFG.baseline_condition:
                            continue

                        effect_key = (
                            f"{param_family}_C(emotion, "
                            f"Treatment('"
                            f"{CFG.baseline_condition}'))"
                            f"[T.{emo}]"
                        )

                        if param_family == 'v':
                            eff = stats.norm.rvs(
                                loc=0, scale=0.5
                            )
                        elif param_family in ['a', 'z']:
                            eff = stats.norm.rvs(
                                loc=0, scale=0.1
                            )
                        elif param_family == 't':
                            eff = stats.norm.rvs(
                                loc=0, scale=0.05
                            )
                        else:
                            eff = 0.0

                        group_truth[effect_key] = float(eff)

        # [FIX-6.3] Generate subject-level parameters with
        # hierarchical variability
        subject_truths = {}
        subjects = self.empirical_df['subj_idx'].unique()

        # Subject-level SD (LP scale) per parameter family
        subj_sd = {'v': 0.5, 'a': 0.15, 't': 0.08, 'z': 0.10}

        for subj in subjects:
            subj_params = {}
            for emo in CFG.emotion_order:
                cell_params = {}
                for p in ['v', 'a', 't', 'z']:
                    # Group-level LP for this condition
                    lp_val = group_truth.get(
                        f"{p}_Intercept", 0.0
                    )
                    if emo != CFG.baseline_condition:
                        effect_key = (
                            f"{p}_C(emotion, "
                            f"Treatment('"
                            f"{CFG.baseline_condition}'))"
                            f"[T.{emo}]"
                        )
                        if effect_key not in group_truth:
                            effect_key = (
                                f"{p}_C(emotion)[T.{emo}]"
                            )
                        lp_val += group_truth.get(
                            effect_key, 0.0
                        )

                    # [FIX-6.3] Add subject-level deviation
                    subj_lp = lp_val + np.random.normal(
                        0, subj_sd[p]
                    )

                    cell_params[p] = self._apply_link_and_clip(
                        p, subj_lp
                    )

                subj_params[emo] = cell_params
            subject_truths[subj] = subj_params

        return group_truth, subject_truths

    # -----------------------------------------------------------------
    # SYNTHETIC DATA GENERATION
    # -----------------------------------------------------------------
    def generate_synthetic_data(
        self,
        subject_truths: Dict[str, Dict[str, Dict[str, float]]],
        iteration: int,
        mode: str
    ) -> pd.DataFrame:
        """
        Constructs synthetic data using HDDM's internal forward
        simulation. Each subject x condition cell uses that
        subject's unique parameters (with hierarchical variability).
        """
        trial_rows = []

        for (subj, emo), group in self.empirical_df.groupby(
            ['subj_idx', 'emotion']
        ):
            n_trials = len(group)
            phys_params = subject_truths[subj][emo]

            # Forward simulation via hddm.generate
            sim_res = gen_rand_data(phys_params, size=n_trials)
            sim_df = (
                sim_res[0]
                if isinstance(sim_res, tuple)
                else sim_res
            )

            row_data = pd.DataFrame({
                'subj_idx': subj,
                'emotion': emo,
                'rt': sim_df['rt'].values,
                'response': sim_df['response'].values
            })

            # Tag true parameters for this cell
            for p in ['v', 'a', 't', 'z']:
                row_data[f'{p}_true'] = phys_params[p]

            trial_rows.append(row_data)

        return pd.concat(trial_rows, ignore_index=True)

    # -----------------------------------------------------------------
    # [FIX-6.4] RECOVERY REFIT WITH CONVERGENCE CHECK
    # -----------------------------------------------------------------
    def execute_recovery_refit(
        self,
        synth_df: pd.DataFrame,
        iteration: int,
        mode: str
    ) -> Tuple[Optional[pd.DataFrame], bool, float]:
        """
        Executes structural refitting of synthetic data.

        Returns:
          summary_df: ArviZ summary (or None on failure)
          converged:  Whether focal R-hat < threshold
          max_rhat:   Maximum R-hat among focal parameters
        """
        synth_df = synth_df.copy()
        synth_df['subj_idx'] = synth_df['subj_idx'].astype(str)

        regressors = _build_regressors(
            self.winning_model, CFG.baseline_condition
        )

        # Always use HDDMRegressor with all 4 params (matching Step 2b)
        full_include = ['v', 'a', 't', 'z']

        model = hddm.HDDMRegressor(
            synth_df,
            regressors,
            include=full_include,
            is_group_model=True,
            group_only_regressors=CFG.group_only_regressors,
            keep_regressor_trace=CFG.keep_regressor_trace,
            informative=CFG.use_informative_priors,
            p_outlier=CFG.p_outlier
        )

        # Unique save path per iteration to avoid file conflicts
        save_prefix = str(
            PATHS['recovery']
            / f"recovery_{self.winning_model}_{mode}_iter{iteration}"
        )

        infdata = model.sample(
            self.recovery_samples,
            burn=self.recovery_burn,
            thin=CFG.default_thin,
            chains=self.recovery_chains,
            return_infdata=True,
            loglike=False,
            ppc=False,
            save_name=save_prefix
        )

        # [FIX-6.4] Convergence check on focal parameters
        summary_df = az.summary(
            infdata, round_to=4, hdi_prob=0.94
        )
        focal_params = CFG.identify_focal_parameters(
            summary_df.index.tolist()
        )
        focal_df = summary_df.loc[
            summary_df.index.isin(focal_params)
        ]

        max_rhat = (
            focal_df['r_hat'].max()
            if (not focal_df.empty
                and 'r_hat' in focal_df.columns)
            else np.nan
        )
        converged = (
            (not np.isnan(max_rhat))
            and (max_rhat <= RECOVERY_RHAT_THRESHOLD)
        )

        # Cleanup: remove per-iteration .nc/.hddm to save disk
        for ext in ['.nc', '.hddm']:
            fpath = Path(f"{save_prefix}{ext}")
            if fpath.exists():
                try:
                    fpath.unlink()
                except OSError:
                    pass

        del model, infdata
        gc.collect()

        return summary_df, converged, float(max_rhat)

    # -----------------------------------------------------------------
    # SINGLE-ITERATION RECOVERY METRICS
    # -----------------------------------------------------------------
    def compute_iteration_metrics(
        self,
        group_truth: Dict[str, float],
        summary_df: pd.DataFrame,
        iteration: int,
        mode: str,
        converged: bool,
        max_rhat: float
    ) -> List[Dict]:
        """
        Computes per-parameter recovery metrics for a single
        iteration: Bias, Squared Error, Coverage, HDI Width.
        """
        records = []

        for param, gt_val in group_truth.items():
            if param not in summary_df.index:
                continue

            rec_row = summary_df.loc[param]
            rec_mean = float(rec_row['mean'])
            hdi_low = float(rec_row['hdi_3%'])
            hdi_high = float(rec_row['hdi_97%'])

            bias = rec_mean - gt_val
            is_covered = hdi_low <= gt_val <= hdi_high

            family = (
                param.split('_')[0] if '_' in param else 'other'
            )
            param_type = (
                'Intercept'
                if 'Intercept' in param
                else 'ConditionEffect'
            )

            records.append({
                'pipeline_hash': self.active_hash,
                'mode': mode,
                'iteration': iteration,
                'Parameter': param,
                'Family': family,
                'Type': param_type,
                'Ground_Truth': gt_val,
                'Recovered_Mean': rec_mean,
                'Bias': bias,
                'Abs_Bias': np.abs(bias),
                'Sq_Error': bias ** 2,
                'HDI_3': hdi_low,
                'HDI_97': hdi_high,
                'HDI_Width': hdi_high - hdi_low,
                'Coverage': int(is_covered),
                'Refit_Converged': converged,
                'Refit_Max_Rhat': max_rhat
            })

        return records

    # -----------------------------------------------------------------
    # [FIX-6.2] INCREMENTAL PERSISTENCE
    # -----------------------------------------------------------------
    @staticmethod
    def _append_records_to_csv(
        records: List[Dict], csv_path: Path
    ):
        """
        Atomically appends records to a CSV file.
        Creates the file with header on first write.
        """
        df_new = pd.DataFrame(records)
        if csv_path.exists():
            df_new.to_csv(
                csv_path, mode='a', header=False, index=False
            )
        else:
            df_new.to_csv(csv_path, index=False)

    # -----------------------------------------------------------------
    # PUBLICATION-READY AGGREGATE VISUALIZATION
    # -----------------------------------------------------------------
    def render_aggregate_diagnostics(self, mode: str):
        """
        Constructs visualizations from ALL converged iterations:
        1. Identity scatter (Ground Truth vs Recovered Mean)
           with marginal distributions.
        2. Coverage rate bar chart per parameter family.
        3. Bias distribution violin per parameter family.
        """
        csv_path = (
            PATHS['recovery']
            / f"recovery_all_iterations_{mode}.csv"
        )
        if not csv_path.exists():
            self.logger.warning(
                f"  [{mode}] No iteration data found. "
                f"Visualization skipped."
            )
            return

        df = pd.read_csv(csv_path)

        # [FIX-6.4] Filter to converged iterations only
        if 'Refit_Converged' in df.columns:
            n_total = df['iteration'].nunique()
            df_conv = df[df['Refit_Converged'] == True]
            n_conv = df_conv['iteration'].nunique()
            self.logger.info(
                f"  [{mode}] Using {n_conv}/{n_total} "
                f"converged iterations for visualization."
            )
        else:
            df_conv = df

        if df_conv.empty:
            self.logger.warning(
                f"  [{mode}] No converged iterations. "
                f"Visualization skipped."
            )
            return

        title_prefix = (
            "Posterior-Anchored"
            if mode == 'posterior'
            else "Prior-Predictive"
        )

        pal = {
            'v': OKABE_ITO[0], 'a': OKABE_ITO[1],
            't': OKABE_ITO[2], 'z': OKABE_ITO[6]
        }

        # --- Plot 1: Identity Scatter (aggregated) ---
        families = df_conv['Family'].unique()
        n_fam = max(1, len(families))
        fig, axes = plt.subplots(
            1, n_fam, figsize=(5 * n_fam, 5), squeeze=False
        )
        axes = axes.flatten()

        for ax, fam in zip(axes, families):
            fam_df = df_conv[df_conv['Family'] == fam]
            color = pal.get(fam, '#333333')

            ax.scatter(
                fam_df['Ground_Truth'],
                fam_df['Recovered_Mean'],
                alpha=0.3, s=20, color=color, edgecolors='none'
            )

            all_vals = pd.concat([
                fam_df['Ground_Truth'],
                fam_df['Recovered_Mean']
            ])
            margin = max(
                0.05, (all_vals.max() - all_vals.min()) * 0.1
            )
            lims = [
                all_vals.min() - margin,
                all_vals.max() + margin
            ]
            ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

            if len(fam_df) >= 3:
                r_val, _ = stats.pearsonr(
                    fam_df['Ground_Truth'],
                    fam_df['Recovered_Mean']
                )
                rmse = float(
                    np.sqrt(fam_df['Sq_Error'].mean())
                )
                ax.set_title(
                    f"{fam.upper()} (r={r_val:.2f}, "
                    f"RMSE={rmse:.3f})",
                    fontweight='bold'
                )
            else:
                ax.set_title(
                    f"{fam.upper()}", fontweight='bold'
                )

            ax.set_xlabel("Ground Truth")
            ax.set_ylabel("Recovered Mean")

        plt.suptitle(
            f"{title_prefix} Recovery (N="
            f"{df_conv['iteration'].nunique()} iterations): "
            f"{self.winning_model.upper()}",
            y=1.05, fontweight='bold'
        )
        sns.despine(trim=True)
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp']
            / f"recovery_scatter_aggregate_"
              f"{self.winning_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        # --- Plot 2: Coverage Rate per Family ---
        coverage_agg = df_conv.groupby('Family').agg(
            Coverage_Rate=('Coverage', 'mean'),
            N=('Coverage', 'count')
        ).reset_index()

        fig, ax = plt.subplots(figsize=(6, 4))
        bars = ax.bar(
            coverage_agg['Family'].str.upper(),
            coverage_agg['Coverage_Rate'],
            color=[
                pal.get(f, '#999999')
                for f in coverage_agg['Family']
            ],
            alpha=0.8, edgecolor='black', linewidth=0.5
        )
        ax.axhline(
            0.94, color='red', linestyle='--', linewidth=1.5,
            label='Nominal 94% Coverage'
        )
        ax.set_ylabel('Empirical Coverage Rate')
        ax.set_title(
            f"{title_prefix}: 94% HDI Coverage",
            fontweight='bold'
        )
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=10)

        # Annotate percentages
        for bar, rate in zip(
            bars, coverage_agg['Coverage_Rate']
        ):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{rate:.0%}",
                ha='center', fontsize=11, fontweight='bold'
            )

        sns.despine(trim=True)
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp']
            / f"recovery_coverage_bar_"
              f"{self.winning_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        # --- Plot 3: Bias Distribution per Family ---
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.violinplot(
            x='Family', y='Bias', data=df_conv,
            palette=pal, inner='box', cut=0, ax=ax,
            alpha=0.6, order=sorted(families)
        )
        ax.axhline(
            0, color='black', linestyle='--',
            linewidth=1.5, alpha=0.7
        )
        ax.set_title(
            f"{title_prefix}: Bias Distribution",
            fontweight='bold'
        )
        ax.set_ylabel('Bias (Recovered - Truth)')
        ax.set_xlabel('Parameter Family')
        sns.despine(trim=True)
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp']
            / f"recovery_bias_violin_"
              f"{self.winning_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        self.logger.info(
            f"  [{mode}] Aggregate visualizations exported."
        )

    # -----------------------------------------------------------------
    # AGGREGATE SUMMARY TABLE
    # -----------------------------------------------------------------
    def export_aggregate_summary(self, mode: str):
        """
        Computes and exports aggregate recovery metrics across
        all converged iterations.
        """
        csv_path = (
            PATHS['recovery']
            / f"recovery_all_iterations_{mode}.csv"
        )
        if not csv_path.exists():
            return

        df = pd.read_csv(csv_path)
        if 'Refit_Converged' in df.columns:
            df = df[df['Refit_Converged'] == True]

        if df.empty:
            return

        agg = df.groupby(['Family', 'Type']).agg(
            N_Iterations=('iteration', 'nunique'),
            N_Params=('Parameter', 'nunique'),
            RMSE=('Sq_Error', lambda x: np.sqrt(x.mean())),
            Mean_Abs_Bias=('Abs_Bias', 'mean'),
            Mean_HDI_Width=('HDI_Width', 'mean'),
            Coverage_Rate=('Coverage', 'mean')
        ).reset_index()

        agg_path = (
            PATHS['tables_supp']
            / f"recovery_aggregate_summary_"
              f"{self.winning_model}_{mode}.csv"
        )
        agg.to_csv(agg_path, index=False)

        # Global correlation
        if len(df) >= 3:
            r_val, p_val = stats.pearsonr(
                df['Ground_Truth'], df['Recovered_Mean']
            )
            rmse_global = float(
                np.sqrt(df['Sq_Error'].mean())
            )
            self.logger.info(
                f"  [{mode}] Global: r={r_val:.3f} "
                f"(p={p_val:.2e}), RMSE={rmse_global:.4f}"
            )

        self.logger.info(f"\n{agg.to_string(index=False)}")

    # -----------------------------------------------------------------
    # [FIX-6.2/6.5/6.6] MASTER ITERATION LOOP
    # -----------------------------------------------------------------
    def run_mode(self, mode: str):
        """
        Executes N iterations of recovery for a given mode
        with per-iteration safety mechanisms.
        """
        self.logger.info(f"\n{'*'*60}")
        self.logger.info(
            f"RECOVERY MODE: {mode.upper()} "
            f"({self.n_iterations} iterations)"
        )
        self.logger.info(f"{'*'*60}")

        csv_path = (
            PATHS['recovery']
            / f"recovery_all_iterations_{mode}.csv"
        )

        # Check for existing results (resume support)
        completed_iters = set()
        if csv_path.exists():
            existing = pd.read_csv(csv_path)
            if 'iteration' in existing.columns:
                completed_iters = set(
                    existing['iteration'].unique()
                )
                self.logger.info(
                    f"  Resume: {len(completed_iters)} "
                    f"iterations already completed."
                )

        consecutive_failures = 0

        for iteration in range(self.n_iterations):
            if iteration in completed_iters:
                continue

            self.logger.info(
                f"\n  --- Iteration {iteration + 1}/"
                f"{self.n_iterations} (mode={mode}) ---"
            )

            # [FIX-6.6] Memory check
            try:
                check_memory_headroom(logger=self.logger)
            except MemoryError:
                self.logger.error(
                    "  Memory insufficient. "
                    "Terminating recovery."
                )
                break

            # [FIX-6.5] Timeout guard
            _set_timeout(ITERATION_TIMEOUT_SEC)

            try:
                t_start = time.time()

                # Phase 1: Generate ground truth + synthetic data
                group_truth, subject_truths = (
                    self.establish_ground_truth(mode, iteration)
                )

                synth_df = self.generate_synthetic_data(
                    subject_truths, iteration, mode
                )

                # Phase 2: Refit
                summary_df, converged, max_rhat = (
                    self.execute_recovery_refit(
                        synth_df, iteration, mode
                    )
                )

                # Phase 3: Compute metrics
                if summary_df is not None:
                    records = self.compute_iteration_metrics(
                        group_truth, summary_df,
                        iteration, mode,
                        converged, max_rhat
                    )

                    # [FIX-6.2] Incremental persistence
                    if records:
                        self._append_records_to_csv(
                            records, csv_path
                        )

                elapsed = (time.time() - t_start) / 60
                status = (
                    'CONVERGED' if converged
                    else f'NOT CONVERGED (R-hat={max_rhat:.3f})'
                )
                self.logger.info(
                    f"  Iteration {iteration + 1}: {status} "
                    f"({elapsed:.1f} min)"
                )

                consecutive_failures = 0

            except TimeoutError:
                _clear_timeout()
                consecutive_failures += 1
                self.logger.warning(
                    f"  Iteration {iteration + 1}: TIMEOUT "
                    f"(>{ITERATION_TIMEOUT_SEC // 60} min). "
                    f"Consecutive failures: "
                    f"{consecutive_failures}/"
                    f"{MAX_CONSECUTIVE_FAILURES}"
                )

            except Exception as e:
                _clear_timeout()
                consecutive_failures += 1
                self.logger.warning(
                    f"  Iteration {iteration + 1}: EXCEPTION "
                    f"({type(e).__name__}: {e}). "
                    f"Consecutive failures: "
                    f"{consecutive_failures}/"
                    f"{MAX_CONSECUTIVE_FAILURES}"
                )

            finally:
                _clear_timeout()
                gc.collect()

            # [FIX-6.5] Consecutive failure guard
            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                self.logger.error(
                    f"  {MAX_CONSECUTIVE_FAILURES} consecutive "
                    f"failures reached. Terminating {mode} mode."
                )
                break

    # -----------------------------------------------------------------
    # FULL PIPELINE
    # -----------------------------------------------------------------
    def run_full_pipeline(self):
        """
        Orchestrates both posterior-anchored and prior-predictive
        recovery protocols with aggregate reporting.
        """
        for mode in ['posterior', 'prior']:
            self.run_mode(mode)
            self.export_aggregate_summary(mode)
            self.render_aggregate_diagnostics(mode)

        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            "DUAL-TRACK PARAMETER RECOVERY COMPLETED"
        )
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    try:
        engine = DualParameterRecoveryEngine()
        engine.run_full_pipeline()
    except Exception as e:
        print(f"\nCRITICAL FAILURE: {e}")
        traceback.print_exc()